In [59]:
from src.module.database import oracle_import


In [60]:
marketplace_mp = oracle_import(
    """select id_, userid, p_date from toki.MONGO_MINIPROGRAMUSERLOGS partition(p_202607)
     where MINIPROGRAMID = '6821b668840548dbe178eacb'"""
)

/workspaces/marketplace-stream-data-recommendation-engine/src/module/database.py:127: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_frame = pd.read_sql(query, engine)


2026-08-19 15:51:27.927 | INFO     | __main__:<module>:1 - Function oracle_import executed in: 2 sec 2 ms


In [61]:
marketplace_mp["USERID"].unique().shape

(1,)

In [62]:
posting_url = 'https://staging-marketplace.toki.mn/ms/catalogue/v1/recommendation'

In [63]:
marketplace_mp.tail()

,ID_,USERID,P_DATE
0,6a561f153df53230e52959c3,615da094e4d670bd57927b0b,20260714
1,6a570a520231f839fdcdc6bf,615da094e4d670bd57927b0b,20260715
2,6a63197a3df53230e52ca07e,615da094e4d670bd57927b0b,20260724


In [64]:
from datetime import datetime, timedelta

cut_date = str(datetime.now().date()-timedelta(days=30))
cut_date = cut_date.replace("-", "")

In [65]:
consumer_events = oracle_import(
    f"select * from toki.marketplace_consumer_EVENTS where p_date >='{cut_date}'"
)

/workspaces/marketplace-stream-data-recommendation-engine/src/module/database.py:127: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_frame = pd.read_sql(query, engine)
2026-08-19 15:51:58.071 | INFO     | __main__:<module>:1 - Function oracle_import executed in: 29 sec 986 ms


In [66]:
from datetime import datetime, timedelta

In [67]:
date_back = datetime.today().date() - timedelta(days=30)
date_back = date_back.strftime("%Y%m%d")

In [68]:
customer_activities = oracle_import(
    "select * from toki.marketplace_consumer_activities where p_date >='{}'".format(cut_date)
)

2026-08-19 15:54:49.741 | INFO     | __main__:<module>:1 - Function oracle_import executed in: 2 min 51 sec 


In [69]:
customer_activities.head()

,ID_,ACTIVITYNAME,ACTIVITYDATA,CREATEDAT,UPDATEDAT,P_DATE
0,6a5d65918c8c31c8476c2525,cart-events,"{'cartId': '67cfcbfca131a1fe55c14645', 'type':...",2026-07-20 08:02:25,2026-07-20 08:02:25,20260720
1,6a5d65918c8c31c8476c2527,cart-events,"{'cartId': '616bc9aa5e7a487833d338bb', 'type':...",2026-07-20 08:02:25,2026-07-20 08:02:25,20260720
2,6a5d65914229463e57f1fc34,cart-events,"{'cartId': '687f79c6f4d95be12e6604d0', 'type':...",2026-07-20 08:02:25,2026-07-20 08:02:25,20260720
3,6a5d6591420fe633e03c9328,cart-events,"{'cartId': '638ebbb30ab58969f2a64bbc', 'type':...",2026-07-20 08:02:25,2026-07-20 08:02:25,20260720
4,6a5d65914229463e57f1fc32,cart-events,"{'cartId': '682d8009dbc8f60bb0234a91', 'type':...",2026-07-20 08:02:25,2026-07-20 08:02:25,20260720


In [70]:
customer_activities["ACTIVITYDATA"].values[0]

"{'cartId': '67cfcbfca131a1fe55c14645', 'type': 'PRODUCT_MODIFIED', 'item': {'productId': '68b7ee190bcb0200c3e8d7f3', 'qty': 1, 'available': True, '_id': '69ddb3b0fde611776a62b121'}, 'cart': {'_id': '68dd0efa175e6115c8470314', 'accountId': '67cfcbfca131a1fe55c14645', 'items': [{'productId': '68febd609494859a95029a76', 'qty': 1, 'available': True, '_id': '6903234d34e745504f059422'}, {'productId': '68b7ee190bcb0200c3e8d7f3', 'qty': 1, 'available': True, '_id': '69ddb3b0fde611776a62b121'}, {'productId': '67a2c3918f15438cb882f463', 'qty': 1, 'available': True, '_id': '6a2fd6cf71348199310a8c41'}], 'createdAt': '2025-10-01T11:22:34.534Z', 'updatedAt': '2026-07-20T00:02:25.932Z'}}"

In [71]:
import ast

In [72]:
ast.literal_eval(customer_activities["ACTIVITYDATA"].values[0])

{'cartId': '67cfcbfca131a1fe55c14645',
 'type': 'PRODUCT_MODIFIED',
 'item': {'productId': '68b7ee190bcb0200c3e8d7f3',
  'qty': 1,
  'available': True,
  '_id': '69ddb3b0fde611776a62b121'},
 'cart': {'_id': '68dd0efa175e6115c8470314',
  'accountId': '67cfcbfca131a1fe55c14645',
  'items': [{'productId': '68febd609494859a95029a76',
    'qty': 1,
    'available': True,
    '_id': '6903234d34e745504f059422'},
   {'productId': '68b7ee190bcb0200c3e8d7f3',
    'qty': 1,
    'available': True,
    '_id': '69ddb3b0fde611776a62b121'},
   {'productId': '67a2c3918f15438cb882f463',
    'qty': 1,
    'available': True,
    '_id': '6a2fd6cf71348199310a8c41'}],
  'createdAt': '2025-10-01T11:22:34.534Z',
  'updatedAt': '2026-07-20T00:02:25.932Z'}}

In [73]:
from datetime import datetime

snapshot_date = datetime.today().date()
snapshot_date = snapshot_date.strftime("%Y%m%d")

In [74]:
marketplace_products = oracle_import(
    f"select * from toki.marketplace_catalogue_products where SNAPSHOT_DATE >= to_date('{snapshot_date}', 'YYYYMMDD')"
)

/workspaces/marketplace-stream-data-recommendation-engine/src/module/database.py:127: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_frame = pd.read_sql(query, engine)


2026-08-19 15:56:29.796 | INFO     | __main__:<module>:1 - Function oracle_import executed in: 1 min 39 sec 


In [75]:
marketplace_products.head()

,ID_,GROUPID,PRODUCTID,STOREID,SKU,BRAND,STOCK,MAINPRICE,SALEPRICE,SALEPERC,SALESTARTAT,SALEENDAT,SALESOPTIONS,KEYWORDS,DISPLAYNAME,DESCRIPTION_,CASHBACKPERC,CASHBACKAMOUNT,MAINOPTIONS,PROPERTY,DISPLAYIMAGEURL,ASSETS,TAXONOMY,URL_,TAXON,VARIANTS,HASGIFT,CREATEDAT,UPDATEDAT,PRODUCTSTATE,SNAPSHOT_DATE
0,6a66d7d838713901eb69c340,461cff20-cc6f-5215-ad7d-9226a538e271,6a38eb8cbf57449d39a95990,6912d444458a4a2f9ca10060,8809934061116,CUCKOO,1,610000,506900,16.9,NaN,NaN,NaN,"['Cuckoo air fryer', '8L air fryer', 'CAF-E082...",Cuckoo 8л багтаамжтай тосгүй шарагч CAF-E0820TB,Cuckoo 8 liter oil-free air fryer model CAF-E0...,0,0.0,"[{'option': 'color', 'value': 'Black'}]","{'resolution': '', 'weight': ''}",https://cdnp.cody.mn/spree/images/3586084/larg...,['https://cdnp.cody.mn/spree/images/3586084/la...,"[{'id': '69fb0b5b9d0fc72657c65255', 'level': 0...",/6a38eb8cbf57449d39a95990,"{'id': '69fb0b5b9d0fc72657c65255', 'label': 'Т...","[{'productId': '6a38eb8cbf57449d39a95990', '_i...",False,2026-07-27 12:00:24,2026-08-13 12:00:18,SETTLED,2026-08-19 00:03:58
1,6a66d7d838713901eb69c341,8e4847ce-b058-5f52-b5b8-47c711138e52,6a38eb8cbf57449d39a95993,6912d444458a4a2f9ca10060,8809934064650,CUCKOO,1,399000,399000,0,NaN,NaN,NaN,"['Cuckoo mixer', 'CFM-J175W', '1000W mixer', '...",Cuckoo 1000Вт хүчин чадалтай холигч CFM-J175W,Cuckoo 1000Вт хүчин чадалтай холигч CFM-J175W.,0,0.0,[],"{'resolution': '', 'weight': ''}",https://cdnp.cody.mn/spree/images/3586092/larg...,['https://cdnp.cody.mn/spree/images/3586092/la...,"[{'id': '69fb0c0347ba407207d05240', 'level': 0...",/6a38eb8cbf57449d39a95993,"{'id': '69fb0c0347ba407207d05240', 'label': 'Г...","[{'productId': '6a38eb8cbf57449d39a95993', '_i...",False,2026-07-27 12:00:24,2026-08-19 00:00:20,ARCHIVED,2026-08-19 00:03:58
2,6a66d7d838713901eb69c342,e3fbc6d5-61e8-5085-8c51-5c8d20a76e58,6a38eb8cbf57449d39a95996,6912d444458a4a2f9ca10060,8809660017371,CUCKOO,1,330000,330000,0,NaN,NaN,NaN,"['CUCKOO hand mixer', 'CFM-I10HW', '400W mixer...",Cuckoo 400Вт хүчин чадалтай гар холигч CFM-I10HW,Cuckoo hand mixer with 400W output power. It w...,0,0.0,[],"{'resolution': '', 'weight': ''}",https://cdnp.cody.mn/spree/images/3586104/larg...,['https://cdnp.cody.mn/spree/images/3586104/la...,"[{'id': '69fb0bec5f98e53690f9fc06', 'level': 0...",/6a38eb8cbf57449d39a95996,"{'id': '69fb0bec5f98e53690f9fc06', 'label': 'А...","[{'productId': '6a38eb8cbf57449d39a95996', '_i...",False,2026-07-27 12:00:24,2026-08-16 18:32:45,SETTLED,2026-08-19 00:03:58
3,6a66d7d838713901eb69c371,cc51eb5f-e573-524b-9d6f-749802dcf3bc,6a3df9516119508a708b0b16,6912d444458a4a2f9ca10060,DW60A6092BB/WT,SAMSUNG,1,2899900,1669900,42.42,NaN,NaN,NaN,"['Samsung dishwasher', 'DW60A6092BB/WT', 'buil...","Samsung 14 сет багтаамж, 7 төрлийн функцтэй нү...",Samsung built-in dishwasher with 14 place sett...,0,0.0,"[{'option': 'color', 'value': 'White'}]","{'resolution': '', 'weight': ''}",https://cdnp.cody.mn/spree/images/3558540/larg...,['https://cdnp.cody.mn/spree/images/3558540/la...,"[{'id': '69fbf9960269cccb2d873552', 'level': 0...",/6a3df9516119508a708b0b16,"{'id': '69fbf9960269cccb2d873552', 'label': 'А...","[{'productId': '6a3df9516119508a708b0b16', '_i...",False,2026-07-27 12:00:24,2026-08-19 00:00:20,ARCHIVED,2026-08-19 00:03:58
4,6a66d7d838713901eb69c372,2d007d4f-fcdc-5760-a316-d6a361d49a34,6a3e318a6119508a708b1204,6912d444458a4a2f9ca10060,ST-9692,WINNING STAR,64,518700,518700,0,NaN,NaN,NaN,"['air fryer', 'Winningstar', 'ST-9692', 'smart...","Winningstar Смарт тосгүй шарагч, 2 тасалгаатай...",type: Тосгүй шарагч capacity: 9000мл control_t...,0,0.0,"[{'option': 'color', 'value': 'Black'}]","{'resolution': '', 'weight': ''}",https://cdnp.cody.mn/spree/images/3561373/larg...,['https://cdnp.cody.mn/spree/images/3561373/la...,"[{'id': '69fb0b5b9d0fc72657c65255', 'level': 0...",/6a3e318a6119508a708b1204,"{'id': '69fb0b5b9d0fc72657c65255', 'label': 'Т...","[{'productId': '6a3e318a6119508a708b1204', '_i...",False,2026-07-27 12:00:24,2026-08-16 00:00:21,SETTLED,2026-08-1

In [76]:
productid = "69fc469bab34c8d11412ec79"
marketplace_products[marketplace_products["PRODUCTID"] == productid]

,ID_,GROUPID,PRODUCTID,STOREID,SKU,BRAND,STOCK,MAINPRICE,SALEPRICE,SALEPERC,SALESTARTAT,SALEENDAT,SALESOPTIONS,KEYWORDS,DISPLAYNAME,DESCRIPTION_,CASHBACKPERC,CASHBACKAMOUNT,MAINOPTIONS,PROPERTY,DISPLAYIMAGEURL,ASSETS,TAXONOMY,URL_,TAXON,VARIANTS,HASGIFT,CREATEDAT,UPDATEDAT,PRODUCTSTATE,SNAPSHOT_DATE
984,6a0b3d3b38713901eb69bc30,86280094-3bd8-58e3-a377-b7f51bd157a2,69fc469bab34c8d11412ec79,6912d444458a4a2f9ca10060,SF321LF0,ROWENTA,1,199900,199900,0,NaN,NaN,NaN,"['Rowenta', 'hair straightener', 'ceramic plat...",Rowenta Үсний индүү SF321LF0,Iron plate: Ceramic\nHeating time: 45sec\nMain...,0,0.0,"[{'option': 'color', 'value': 'Black'}]","{'resolution': '', 'weight': ''}",https://cdnp.cody.mn/spree/images/3431242/larg...,['https://cdnp.cody.mn/spree/images/3431242/la...,"[{'id': '69fb16a36712683b9c3f5b84', 'level': 0...",/69fc469bab34c8d11412ec79,"{'id': '69fb16a36712683b9c3f5b84', 'label': 'Ү...","[{'productId': '69fc469bab34c8d11412ec79', '_i...",False,2026-05-19 00:24:26,2026-08-19 00:00:20,ARCHIVED,2026-08-19 00:03:58


In [77]:
consumer_events.shape

(384271, 11)

In [78]:
consumer_events.tail()

,ID_,EVENTNAME,EVENTVALUE,ACCOUNTID,SESSIONID,TIMESTAMP_,USERAGENT,URL_,CREATEDAT,UPDATEDAT,P_DATE
384266,6a8483b38c8c31c847840e9f,product_click,"{'productIds': ['699e693efd48a0ea61c5e444', '6...",5f71a56427f18c5dd9f2adb7,S_HO7YG4HTNJ7M5YB_wyRnsHM7s07qeG,2026-08-18T16:09:23.500Z,Mozilla/5.0 (Linux; Android 16; SM-F956B Build...,https://marketplace.toki.mn/search/search/S25,2026-08-19 00:09:23,2026-08-19 00:09:23,20260819
384267,6a8483b6420fe633e0547766,taxon_click,{'taxon': {'label': 'Гар утас'}},63ce0ab10197514aee468d19,aeZB6JRZun8jbRr_8_jhOiSocbWPNfPm,2026-08-18T16:11:01.922Z,Mozilla/5.0 (iPhone; CPU iPhone OS 18_7 like M...,https://marketplace.toki.mn/home/674429ce8f734...,2026-08-19 00:09:26,2026-08-19 00:09:26,20260819
384268,6a8483b94229463e5709d95f,taxon_click,{'taxon': {'label': 'Зөөврийн компьютер'}},628c57bb9351cf448ab40ca5,CUa1R6INyPI7mxjyv9otOKqol4-K_Teo,2026-08-18T16:09:29.060Z,Mozilla/5.0 (Linux; Android 16; SM-S928N Build...,https://marketplace.toki.mn/home/67442a8e5021c...,2026-08-19 00:09:29,2026-08-19 00:09:29,20260819
384269,6a8483bd8c8c31c847840ff7,product_click,"{'productIds': ['6a309f8fd46aca65f808449f'], '...",68b446d9854ddf7fec61fee9,bk2Ju_7bBlMy5TmHng3agK4bPfltF0i5,2026-08-18T16:09:33.713Z,Mozilla/5.0 (iPhone; CPU iPhone OS 18_7 like M...,https://marketplace.toki.mn/home/674424ac6134f...,2026-08-19 00:09:33,2026-08-19 00:09:33,20260819
384270,6a8483bdce31add3c36e231e,product_click,"{'productIds': ['699e680dfd48a0ea61c5e43f', '6...",5f71a56427f18c5dd9f2adb7,S_HO7YG4HTNJ7M5YB_wyRnsHM7s07qeG,2026-08-18T16:09:33.524Z,Mozilla/5.0 (Linux; Android 16; SM-F956B Build...,https://marketplace.toki.mn/search/search/S25,2026-08-19 00:09:33,2026-08-19 00:09:33,20260819


In [79]:
consumer_events.head(2)

,ID_,EVENTNAME,EVENTVALUE,ACCOUNTID,SESSIONID,TIMESTAMP_,USERAGENT,URL_,CREATEDAT,UPDATEDAT,P_DATE
0,6a5df676805151b025e94faf,product_click,"{'productIds': ['69c644e15a6a340632dfd5ba', '6...",5f8028279448b1fdd848af37,_ke14Nnfdy9UgroSfWD00iNzozm52v-F,2026-07-20T10:20:38.240Z,Mozilla/5.0 (iPhone; CPU iPhone OS 18_6_2 like...,https://marketplace.toki.mn/home/674429ce8f734...,2026-07-20 18:20:38,2026-07-20 18:20:38,20260720
1,6a5df67c8c8c31c8476c851f,product_click,"{'productIds': ['67f8677d941590a9221302d0', '6...",682f06f3564e818fb30f5d4d,RBMIhheKxJUliLvtKuJGsosr-Zo3vQb0,2026-07-20T10:20:44.925Z,Mozilla/5.0 (Linux; Android 16; SM-A556E Build...,https://marketplace.toki.mn/home/674429ce8f734...,2026-07-20 18:20:44,2026-07-20 18:20:44,20260720


In [80]:
consumer_events.groupby("EVENTNAME").count()

,ID_,EVENTVALUE,ACCOUNTID,SESSIONID,TIMESTAMP_,USERAGENT,URL_,CREATEDAT,UPDATEDAT,P_DATE
EVENTNAME,,,,,,,,,,
product_click,168007,168007,168007,166349,168007,168007,168007,168007,168007,168007
taxon_click,216264,210612,216264,214212,216264,216264,216264,216264,216264,216264


In [81]:
consumer_events.head(2).to_json(orient="records")

/tmp/ipykernel_268242/4275074210.py:1: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  consumer_events.head(2).to_json(orient="records")


'[{"ID_":"6a5df676805151b025e94faf","EVENTNAME":"product_click","EVENTVALUE":"{\'productIds\': [\'69c644e15a6a340632dfd5ba\', \'69c644e25a6a340632dfd5bd\', \'69c644e35a6a340632dfd5c0\', \'69f853bb473e3021fa71c759\'], \'taxon\': {\'label\': \'\\u0413\\u0430\\u0440 \\u0443\\u0442\\u0430\\u0441\'}}","ACCOUNTID":"5f8028279448b1fdd848af37","SESSIONID":"_ke14Nnfdy9UgroSfWD00iNzozm52v-F","TIMESTAMP_":"2026-07-20T10:20:38.240Z","USERAGENT":"Mozilla\\/5.0 (iPhone; CPU iPhone OS 18_6_2 like Mac OS X) AppleWebKit\\/605.1.15 (KHTML, like Gecko) Mobile\\/15E148","URL_":"https:\\/\\/marketplace.toki.mn\\/home\\/674429ce8f73495d25242f69","CREATEDAT":1784571638000,"UPDATEDAT":1784571638000,"P_DATE":"20260720"},{"ID_":"6a5df67c8c8c31c8476c851f","EVENTNAME":"product_click","EVENTVALUE":"{\'productIds\': [\'67f8677d941590a9221302d0\', \'67f86797a92a96984b391f12\', \'67f867ab1ca4873792406de3\', \'68b7ee190bcb0200c3e8d803\'], \'taxon\': {\'label\': \'\\u0413\\u0430\\u0440 \\u0443\\u0442\\u0430\\u0441\'}}",

In [82]:
consumer_events[consumer_events["EVENTNAME"] == "taxon_click"]["EVENTVALUE"].unique()

<StringArray>
[                                                                                                                                '{'taxon': {'label': 'Гар утас'}}',
                                                                                                                                   '{'taxon': {'label': 'Зурагт'}}',
                                                                                                                                '{'taxon': {'label': 'Компьютер'}}',
                                                                                                                          '{'taxon': {'label': 'Тренд технологи'}}',
                                                                                                                                   '{'taxon': {'label': 'Чихэвч'}}',
                                                                                       '{'productIds': ['6a0546a8109dd372dc9c0237'], 'taxon': {'label': 'Gaming'}

In [83]:
consumer_events[consumer_events["ACCOUNTID"] == "6a5e47214aeec353171ccaa0"]

,ID_,EVENTNAME,EVENTVALUE,ACCOUNTID,SESSIONID,TIMESTAMP_,USERAGENT,URL_,CREATEDAT,UPDATEDAT,P_DATE


In [84]:
consumer_events[consumer_events["SESSIONID"] == "jPAaTyDWFjD1JsHyR0ux3hewNYRvNvRy"][
    "EVENTVALUE"
].values

<StringArray>
[]
Length: 0, dtype: str

In [85]:
customer_activities["ACTIVITYDATA"].values[0]

"{'cartId': '67cfcbfca131a1fe55c14645', 'type': 'PRODUCT_MODIFIED', 'item': {'productId': '68b7ee190bcb0200c3e8d7f3', 'qty': 1, 'available': True, '_id': '69ddb3b0fde611776a62b121'}, 'cart': {'_id': '68dd0efa175e6115c8470314', 'accountId': '67cfcbfca131a1fe55c14645', 'items': [{'productId': '68febd609494859a95029a76', 'qty': 1, 'available': True, '_id': '6903234d34e745504f059422'}, {'productId': '68b7ee190bcb0200c3e8d7f3', 'qty': 1, 'available': True, '_id': '69ddb3b0fde611776a62b121'}, {'productId': '67a2c3918f15438cb882f463', 'qty': 1, 'available': True, '_id': '6a2fd6cf71348199310a8c41'}], 'createdAt': '2025-10-01T11:22:34.534Z', 'updatedAt': '2026-07-20T00:02:25.932Z'}}"

In [86]:
customer_activities["ACTIVITYDATA"].values[2]

"{'cartId': '687f79c6f4d95be12e6604d0', 'type': 'PRODUCT_MODIFIED', 'item': {'productId': '6809a24a1ca487379240e239', 'qty': 1, 'available': True, '_id': '6a054ea836d1ad7dc4176e59'}, 'cart': {'_id': '68dc7114175e6115c8464750', 'accountId': '687f79c6f4d95be12e6604d0', 'items': [{'productId': '6809a24a1ca487379240e239', 'qty': 1, 'available': True, '_id': '6a054ea836d1ad7dc4176e59'}], 'createdAt': '2025-10-01T00:08:52.487Z', 'updatedAt': '2026-07-20T00:02:25.937Z'}}"

In [87]:
customer_activities["ACTIVITYNAME"].unique()

<StringArray>
['cart-events', 'limit-events', 'order-events', 'wishlist-events']
Length: 4, dtype: str

In [88]:
customer_activities.groupby(["ACTIVITYNAME"]).count()

,ID_,ACTIVITYDATA,CREATEDAT,UPDATEDAT,P_DATE
ACTIVITYNAME,,,,,
cart-events,3547582,3547582,3547582,3547582,3547582
limit-events,47446,47446,47446,47446,47446
order-events,16966,16966,16966,16966,16966
wishlist-events,2422,2422,2422,2422,2422


In [89]:
customer_activities[customer_activities["ACTIVITYNAME"] == "cart-events"].head()[
    "ACTIVITYDATA"
].values[4]

"{'cartId': '682d8009dbc8f60bb0234a91', 'type': 'PRODUCT_MODIFIED', 'item': {'productId': '68b7ee190bcb0200c3e8d7f3', 'qty': 1, 'available': True, '_id': '691a9de334e745504f3c5323'}, 'cart': {'_id': '68dd0dda175e6115c84700c0', 'accountId': '682d8009dbc8f60bb0234a91', 'items': [{'productId': '68b7ee190bcb0200c3e8d7f3', 'qty': 1, 'available': True, '_id': '691a9de334e745504f3c5323'}], 'createdAt': '2025-10-01T11:17:46.538Z', 'updatedAt': '2026-07-20T00:02:25.922Z'}}"

In [90]:
customer_activities["ACTIVITYDATA"].values[0]

"{'cartId': '67cfcbfca131a1fe55c14645', 'type': 'PRODUCT_MODIFIED', 'item': {'productId': '68b7ee190bcb0200c3e8d7f3', 'qty': 1, 'available': True, '_id': '69ddb3b0fde611776a62b121'}, 'cart': {'_id': '68dd0efa175e6115c8470314', 'accountId': '67cfcbfca131a1fe55c14645', 'items': [{'productId': '68febd609494859a95029a76', 'qty': 1, 'available': True, '_id': '6903234d34e745504f059422'}, {'productId': '68b7ee190bcb0200c3e8d7f3', 'qty': 1, 'available': True, '_id': '69ddb3b0fde611776a62b121'}, {'productId': '67a2c3918f15438cb882f463', 'qty': 1, 'available': True, '_id': '6a2fd6cf71348199310a8c41'}], 'createdAt': '2025-10-01T11:22:34.534Z', 'updatedAt': '2026-07-20T00:02:25.932Z'}}"

In [91]:
# there 4 events that could potentially be used to track user activity: "view_product"
# limit-events - user checked the lease limit
# order-events - user placed an order or completed an order
# wishlist-events - user added a product to their wishlist
# card-events added card or removed card, modified in the cards etc events,

In [92]:
from src.database import pgsql_import

master_catalog_profile = pgsql_import(
    "select * from marketplace_catalog_data_extended_version3"
)

In [93]:
import pandas as pd

pd.set_option("display.max_columns", 100)

In [94]:
master_catalog_profile.head(2)

,carried_located_in,main_category,sub_category,product_category,exact_product_category,manufacturer,generic_name,actual_product,size,power_consumption,year,specifications,sku,connectivity,stock,price,dimensions,index,product_id,shop_name,discount,main_option,details,url_link,keywords,best_used_for,premium_grade,price_range,insurance,delivery,taxon_id,taxon_name,description,best_used_for_detail,taxondict,prompt_,created_at,colors,image_list,images,mainoption,data_relation_map,color,image,productstate,createdat,updatedat,productmeta,saleprice,image_urls,details_translation,group_id
0,Living Room,Electronics,Televisions,Mini LED QLED 4K TVs,Sony 85XR50 85-inch Mini LED QLED 4K HDR Googl...,SONY,Smart Television,Mini LED QLED 4K HDR Google 85 inch tv /SONY-K...,85 inches,,2026,"{""resolution"": ""UHD 4K 3840x2160p"", ""processor...",SONY-K-85XR50,"{""wifi"": ""Yes"", ""bluetooth"": ""Yes"", ""hdmi"": ""4...",2,11999900 MNT,Without stand: 1891 x 1085 x 492 mm; With stan...,shop_0_1772709420,6968a70895de95f954a9a16c,BSB Electronics,"{""regular_price"": ""12999900 MNT"", ""sale_price""...","{""screen-size"": ""85inch"", ""resolution"": ""3840x...",85-inch Sony Mini LED QLED 4K HDR Google TV wi...,https://imagedelivery.net/jUCGGlEY6TCUtkJb-Fl1...,"[""Sony TV"", ""85 inch TV"", ""Mini LED"", ""QLED"", ...",Entertainment,premium,luxury,,"Pickup, Shipping",674425c2b07fff9ff4a48d4c,tv,Screen size: 85 inch Resolution: UHD 4K 3840x2...,Optimized for home entertainment and streaming...,NaN,"{""role"": ""user"", ""content"": ""product index : 0...",2026-03-05 19:17:48.739441,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9769a9ad-9471-5347-abf4-c920d69a0e3e
1,Living Room,Electronics,Televisions,4K UHD TVs,Full Array LED 4K HDR Smart TV,Panasonic,Smart Television,Panasonic TH-75NX900,75 inch,,2026,"{""screen_type"": ""Full Array LED"", ""resolution""...",PANA-TH-75NX900M,"{""wifi"": ""Yes"", ""bluetooth"": ""Bluetooth 5.1"", ...",1,4599900 MNT,1675 x 1041 x 363 mm,shop_1_1772709420,6968a70895de95f954a9a16f,BSB Electronics,"{""regular_price"": ""5499900 MNT"", ""sale_price"":...","{""screen-size"": ""75 inch"", ""resolution"": ""3840...",PRODUCTSTATE: ARCHIVED; SYNCSTATE: SYNCED; CRE...,https://imagedelivery.net/jUCGGlEY6TCUtkJb-Fl1...,"[""Panasonic"", ""Panasonic TH-75NX900"", ""75 inch...",Entertainment,premium,high-end,,Pickup,674425c2b07fff9ff4a48d4c,tv,"Screen size: 75 inch, 1675mm Screen: Full Arra...",Ideal for home entertainment and streaming (mo...,NaN,"{""role"": ""user"", ""content"": ""product index : 1...",2026-03-05 19:18:43.058787,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,eae2fd2c-1c51-56f4-a7ec-7561dbfd11e9


In [95]:
master_catalog_profile.tail(2).to_json(orient="records")

'[{"carried_located_in":"On Person","main_category":"Electronics","sub_category":"Smartphones","product_category":"Android Phones","exact_product_category":"Huawei Pura 90s Pro","manufacturer":"Huawei","generic_name":"Smartphone","actual_product":"Huawei Pura 90s Pro","size":"256GB","power_consumption":"","year":"0","specifications":"{\\"color\\": \\"Mulberry Black\\", \\"capacity\\": \\"256GB\\", \\"inventory_type\\": \\"MOBILEPHONE\\", \\"sales_channel\\": \\"ALL\\", \\"product_state\\": \\"MODIFIED\\"}","sku":"51098YXL","connectivity":"{}","stock":"23","price":"4298000.0 MNT","dimensions":"","index":"shop_0_1787111203","product_id":"6a8505914b18b6a1f54fa1e0","shop_name":"TOKI","discount":"{\\"regular_price\\": \\"4298000.0 MNT\\"}","main_option":"{\\"storage\\": \\"256GB\\", \\"color\\": \\"Mulberry Black\\"}","details":"PPSKUID: YNY-TEC-HUAP90PX6I. Updated at 2026-08-19T03:37:03.535Z. Available stock across branches totals 23 units.","url_link":"https:\\/\\/upload-web.toki.mn\\/upl

In [96]:
import pandas as pd

pd.set_option("display.max_columns", 100)

In [97]:
print(master_catalog_profile.head(2).to_json(orient="records"))

[{"carried_located_in":"Living Room","main_category":"Electronics","sub_category":"Televisions","product_category":"Mini LED QLED 4K TVs","exact_product_category":"Sony 85XR50 85-inch Mini LED QLED 4K HDR Google TV","manufacturer":"SONY","generic_name":"Smart Television","actual_product":"Mini LED QLED 4K HDR Google 85 inch tv \/SONY-K-85XR50\/","size":"85 inches","power_consumption":"","year":"2026","specifications":"{\"resolution\": \"UHD 4K 3840x2160p\", \"processor\": \"XR Processor (image enhancement AI)\", \"panel_type\": \"Mini LED QLED\", \"wide_viewing\": \"X-Wide Angle\", \"anti_reflection\": \"X-Anti Reflection\", \"operating_system\": \"ANDROID (Google TV)\", \"preinstalled_apps\": [\"Netflix\", \"Prime Video\", \"Disney+\", \"YouTube\", \"Apple TV\"], \"ai_image_enhancement\": \"Yes\", \"refresh_rate\": \"Motionflow\u2122 XR 800Hz (Native 120Hz)\", \"eye_protection\": \"Yes\", \"audio\": [\"Dolby Atmos\", \"DSEE\", \"Cinema\", \"X-Balanced Speaker\"], \"hdmi_ports\": 4, \"

## Recommendation Engine — Live API Integration

The engine is running on **port 8018** (local) and publicly via Cloudflare Tunnel.

### Available endpoint
| Endpoint | Method | Auth | Purpose |
|---|---|---|---|
| `/api/v1/events` | POST | ✓ | Ingest `customer_activities` events (order, cart, limit, wishlist, view, product_click, taxon_click) |
| `/api/v1/consumer-events` | POST | ✓ | Ingest Oracle `consumer_events` rows directly (EVENTNAME, EVENTVALUE, ACCOUNTID …) |
| `/api/v1/infer` | POST | ✓ | On-demand single-taxon inference for a user |
| `/api/v1/feed` | POST | ✓ | Multi-taxon feed — returns top-N products per taxon; auto-pushes to marketplace in background |
| `/api/v1/feed/push` | POST | ✓ | Generate feed AND synchronously push to the shop's endpoint; returns push status |
| `/api/v1/recommendations/taxon` | POST | ✓ | Taxon/category page product grid (CBF 45% + CF 30% + Pop 25%) |
| `/api/v1/recommendations/product` | POST | ✓ | Product detail page "You may also like" panel (CBF 60% + CF 40%) |
| `/api/v1/recommendations/basket` | POST | ✓ | Basket page cross-sell panel (CBF 40% + CF 35% + Pop 25%) |
| `/api/v1/health` | GET | — | Liveness check |
| `/api/v1/catalog/status` | GET | — | Catalog index stats |
| `/api/v1/metrics` | GET | — | Live ingestion/recommendation metrics snapshot |
| `/api/v1/logs/ingest` | GET | — | Recent ingest batch log |
| `/api/v1/logs/push` | GET | — | Recent marketplace push log |
| `/dashboard` | GET | — | Live HTML monitoring dashboard |

### Marketplace push payload — `POST https://staging-marketplace.toki.mn/ms/catalogue/v1/recommendation`
```json
{
  "accountId": "<account_id>",
  "products": [
    { "productId": "6a068636e452e1b264f6e6c6", "taxonId": "674425c2b07fff9ff4a48d4c" },
    { "productId": "69fc469bab34c8d11412ec79", "taxonId": "69fbef9bda75a61ceadc7607" }
  ]
}
```

```bash
curl --location 'https://staging-marketplace.toki.mn/ms/catalogue/v1/recommendation' \
  --header 'Content-Type: application/json' \
  --data '{
    "products": [
      { "productId": "6a068636e452e1b264f6e6c6", "taxonId": "674425c2b07fff9ff4a48d4c" }
    ],
    "accountId": "64b7b484fa4f99d010979ea0"
  }'
```

> **Note:** `accountId` must be a 24-char hex MongoDB ObjectId. Synthetic/test IDs are skipped by the push validator.


In [98]:
import requests

BASE = "http://localhost:8018"
HEADERS = {"Content-Type": "application/json"}

# ── Catalog status ─────────────────────────────────────────────────────────────
status = requests.get(f"{BASE}/api/v1/catalog/status").json()
print(f"Catalog: {status['catalog_size']} products | TF-IDF {status['tfidf_shape']}")
print(
    f"Taxon labels mapped: {status['taxon_label_map_size']} | Taxon slugs: {status['taxon_name_map_size']}"
)
print(
    f"Active sessions: {status['active_sessions']} | Tracked users: {status['tracked_users']}"
)


Catalog: 4207 products | TF-IDF [4207, 30000]
Taxon labels mapped: 165 | Taxon slugs: 79
Active sessions: 0 | Tracked users: 0


In [99]:
sample_consumer_rows = [
    {
        "ID_": "6a44ce39ce31add3c347e3d6",
        "EVENTNAME": "product_click",
        "EVENTVALUE": "{'productIds': ['69fc469bab34c8d11412ec79'], 'taxon': {'label': 'Үсний хэрэгсэл'}}",
        "ACCOUNTID": "66fbc5824e022311128232ae",
        "SESSIONID": "jPAaTyDWFjD1JsHyR0ux3hewNYRvNvRy",
        "TIMESTAMP_": "2026-07-01T08:21:23.894Z",
        "USERAGENT": "Mozilla/5.0 (iPhone; CPU iPhone OS 18_7 like Mac OS X) AppleWebKit/605.1.15",
    },
    {
        "ID_": "6a44ce3b420fe633e02e2e78",
        "EVENTNAME": "taxon_click",
        "EVENTVALUE": "{'taxon': {'label': 'Гар утас'}}",
        "ACCOUNTID": "5ff870ee4f636263bd482270",
        "SESSIONID": "2xI3rxpGJbBOeY4vnY_1EBSkTuAL8Pf9",
        "TIMESTAMP_": "2026-07-01T08:22:19.413Z",
        "USERAGENT": "Mozilla/5.0 (iPhone; CPU iPhone OS 18_6 like Mac OS X) AppleWebKit/605.1.15",
    },
]

r = requests.post(f"{BASE}/api/v1/consumer-events", json={"events": sample_consumer_rows})
result = r.json()
print(f"Status: {result['status']} | processed={result['processed']} failed={result['failed']}")
print()
for rec in result["recommendations"]:
    print(f"  user: {rec['id']}")
    print(f"  taxon_id: {rec['taxon_id']}")
    print(f"  strategy: {rec['strategy']} | intent: {rec['intent_score']} | device: {rec['device']}")
    print(f"  recs ({rec['count']}): {rec['recommendations'][:4]}...")
    print()


Status: accepted | processed=2 failed=0

  user: 66fbc5824e022311128232ae
  taxon_id: 69fb16a36712683b9c3f5b84
  strategy: hybrid | intent: 1.5 | device: mobile
  recs (12): ['69fc469bab34c8d11412ec76', '69fc469bab34c8d11412ec7f', '69fc4699ab34c8d11412ebf6', '69fc469cab34c8d11412eca2']...

  user: 5ff870ee4f636263bd482270
  taxon_id: None
  strategy: popular | intent: 0.5 | device: mobile
  recs (1): ['69fc469bab34c8d11412ec79']...



In [100]:
# ── POST /feed: multi-taxon recommendations ────────────────────────────────────
# account_id must be a 24-char hex MongoDB ObjectId for shop push to succeed
DEMO_ACCOUNT = "66fbc5824e022311128232ae"

requests.post(
    f"{BASE}/api/v1/events",
    json={
        "events": [
            {
                "account_id": DEMO_ACCOUNT,
                "activity_name": "product_click",
                "activity_data": {
                    "productIds": ["69fc469bab34c8d11412ec79"],
                    "taxon": {"label": "household-appliances-multi-purpose-vacuum"},
                },
            },
            {
                "account_id": DEMO_ACCOUNT,
                "activity_name": "order-events",
                "activity_data": {
                    "accountid": DEMO_ACCOUNT,
                    "productid": "698977503516dac1b3e97a6c",
                    "action": "complete",
                },
            },
            {
                "account_id": DEMO_ACCOUNT,
                "activity_name": "limit-events",
                "activity_data": {
                    "accountid": DEMO_ACCOUNT,
                    "limit_amount": 3000000,
                    "currency": "MNT",
                },
            },
            {
                "account_id": DEMO_ACCOUNT,
                "activity_name": "wishlist-events",
                "activity_data": {
                    "accountid": DEMO_ACCOUNT,
                    "productid": "6a309f8bd46aca65f8084431",
                    "action": "add",
                },
            },
        ]
    },
)

r = requests.post(
    f"{BASE}/api/v1/feed",
    json={"account_id": DEMO_ACCOUNT, "top_taxons": 3, "top_n_per_taxon": 8},
)
feed = r.json()
print(f"Strategy: {feed['strategy']} | Intent: {feed['intent_score']} | Total products: {feed['total_products']}")
print()
for tf in feed["taxon_feeds"]:
    print(f"  [{tf['taxon_name']}]")
    print(f"   taxon_id: {tf['taxon_id']}")
    print(f"   recommendations ({tf['count']}): {tf['recommendations'][:4]}...")
    print()


Strategy: multi_taxon_hybrid | Intent: 15.0 | Total products: 24

  [household-appliances-vacuum]
   taxon_id: 69fb128116b9019c3408d99a
   recommendations (8): ['6a05469d109dd372dc9bfc96', '6a02dea5e96e1eeb38ff74bf', '6a0546a1109dd372dc9bfe37', '6a0546aa109dd372dc9c02bb']...

  [gaming-console-accessory]
   taxon_id: 698c4044e783dbd39ed224f6
   recommendations (8): ['69fb11292a970a555ee2431a', '6a054699109dd372dc9bfb31', '6989774f3516dac1b3e979ee', '6a0c50f2cb69d0907167363e']...

  [household-appliances-multi-purpose-vacuum]
   taxon_id: 69fb126eae2e0da5c8bca3a0
   recommendations (8): ['69fc469bab34c8d11412ec4a', '6a0546a5109dd372dc9c005b', '6a054699109dd372dc9bfb34', '6a05469f109dd372dc9bfd65']...



In [101]:
# ── POST /feed/push: generate feed and push to staging marketplace ─────────────
# accountId must be 24-char hex — validated before push to avoid shop 400 errors
r = requests.post(
    f"{BASE}/api/v1/feed/push",
    json={
        "account_id": DEMO_ACCOUNT,
        "top_taxons": 3,
        "top_n_per_taxon": 10,
        "shop_feed_url": posting_url,
        "push_timeout_seconds": 3.0,
    },
)
push_result = r.json()
print(f"Strategy: {push_result['strategy']} | Total products: {push_result['total_products']}")
print(f"Push status: {push_result['push_status']} | Push URL: {push_result['push_url']}")
if push_result["push_error"]:
    print(f"Push error: {push_result['push_error']}")
print()
for tf in push_result["taxon_feeds"]:
    print(f"  {tf['taxon_name'] or tf['taxon_id']}: {tf['count']} products")


Strategy: multi_taxon_hybrid | Total products: 28
Push status: ok | Push URL: https://staging-marketplace.toki.mn/ms/catalogue/v1/recommendation

  computer-audio-video-accessory: 10 products
  televisions-home-theatre: 10 products
  computer-gaming-gear-accessory: 9 products


In [102]:
# ── Preview: exact payload that will be sent to the marketplace ────────────────
# Generates the feed locally and prints the full products list before pushing
preview_feed = requests.post(
    f"{BASE}/api/v1/feed",
    json={"account_id": DEMO_ACCOUNT, "top_taxons": 5, "top_n_per_taxon": 10},
).json()

preview_payload = {
    "accountId": DEMO_ACCOUNT,
    "products": [
        {"productId": pid, "taxonId": tf["taxon_id"]}
        for tf in preview_feed.get("taxon_feeds", [])
        for pid in tf.get("recommendations", [])
    ],
}

print(f"accountId : {preview_payload['accountId']}")
print(f"products  : {len(preview_payload['products'])} items")
print()
for i, p in enumerate(preview_payload["products"]):
    print(f"  [{i+1:>2}]  productId: {p['productId']}   taxonId: {p['taxonId']}")


accountId : 66fbc5824e022311128232ae
products  : 49 items

  [ 1]  productId: 694369d1d772ddc6a3db89d5   taxonId: 698c3cb3e783dbd39ed224eb
  [ 2]  productId: 694369d1d772ddc6a3db8984   taxonId: 698c3cb3e783dbd39ed224eb
  [ 3]  productId: 698977463516dac1b3e9750b   taxonId: 698c3cb3e783dbd39ed224eb
  [ 4]  productId: 694369d1d772ddc6a3db89db   taxonId: 698c3cb3e783dbd39ed224eb
  [ 5]  productId: 694369d1d772ddc6a3db8954   taxonId: 698c3cb3e783dbd39ed224eb
  [ 6]  productId: 694369d1d772ddc6a3db897e   taxonId: 698c3cb3e783dbd39ed224eb
  [ 7]  productId: 6989774f3516dac1b3e979b2   taxonId: 698c3cb3e783dbd39ed224eb
  [ 8]  productId: 698977473516dac1b3e975b0   taxonId: 698c3cb3e783dbd39ed224eb
  [ 9]  productId: 6989774f3516dac1b3e979a3   taxonId: 698c3cb3e783dbd39ed224eb
  [10]  productId: 698977483516dac1b3e97601   taxonId: 698c3cb3e783dbd39ed224eb
  [11]  productId: 6a05469d109dd372dc9bfcb4   taxonId: 674425d319948cc421e4717b
  [12]  productId: 6a0546ab109dd372dc9c0367   taxonId: 674425

In [103]:
# ── Check ingest + push logs ───────────────────────────────────────────────────
import json

ingest_log = requests.get(f"{BASE}/api/v1/logs/ingest?limit=5").json()
push_log   = requests.get(f"{BASE}/api/v1/logs/push?limit=5").json()

print("=== Recent Ingest Batches ===")
for e in ingest_log["entries"]:
    print(f"  {e['ts'][:19]}  [{e['source']}]  processed:{e['processed']}  failed:{e['failed']}  users:{len(e['users'])}  {e['event_types']}")

print()
print("=== Recent Marketplace Pushes ===")
for e in push_log["entries"]:
    status_str = "✓" if e["push_status"] == "ok" else ("↷" if e["push_status"] == "skipped" else "✗")
    print(f"  {e['ts'][:19]}  {status_str} [{e['push_status']}]  account:{e['account_id']}  products:{e['products_count']}")
    if e.get("push_error"):
        print(f"    error: {e['push_error']}")


=== Recent Ingest Batches ===
  2026-08-19T07:56:32  [events]  processed:4  failed:0  users:1  {'product_click': 1, 'order-events': 1, 'limit-events': 1, 'wishlist-events': 1}
  2026-08-19T07:56:32  [consumer-events]  processed:2  failed:0  users:2  {'product_click': 1, 'taxon_click': 1}
  2026-08-19T02:54:21  [events]  processed:1  failed:0  users:1  {'view_product': 1}
  2026-08-19T02:54:21  [events]  processed:1  failed:0  users:1  {'view_product': 1}
  2026-08-19T02:54:21  [consumer-events]  processed:1  failed:0  users:1  {'product_click': 1}

=== Recent Marketplace Pushes ===
  2026-08-19T07:56:33  ✓ [ok]  account:5ff870ee4f636263bd482270  products:30
  2026-08-19T07:56:32  ✓ [ok]  account:66fbc5824e022311128232ae  products:30
  2026-08-19T07:56:32  ✓ [ok]  account:66fbc5824e022311128232ae  products:29
  2026-08-19T07:56:32  ✓ [ok]  account:66fbc5824e022311128232ae  products:24
  2026-08-19T07:56:32  ✓ [ok]  account:66fbc5824e022311128232ae  products:10


In [104]:
# ── Public URL (internet-exposed via Cloudflare Tunnel) ───────────────────────
# Share this URL with shop developers for testing
PUBLIC_URL = "http://10.22.4.13:8018"

print("=== Public API endpoints for shop developers ===")
print()
print(f"Docs/Swagger UI:       {PUBLIC_URL}/docs")
print(f"Monitoring dashboard:  {PUBLIC_URL}/dashboard")
print(f"Health check:     GET  {PUBLIC_URL}/api/v1/health")
print(f"Catalog status:   GET  {PUBLIC_URL}/api/v1/catalog/status")
print(f"Metrics:          GET  {PUBLIC_URL}/api/v1/metrics")
print()
print(f"Ingest events:    POST {PUBLIC_URL}/api/v1/events")
print(f"Consumer events:  POST {PUBLIC_URL}/api/v1/consumer-events")
print(f"Single-taxon infer:POST {PUBLIC_URL}/api/v1/infer")
print(f"Multi-taxon feed: POST {PUBLIC_URL}/api/v1/feed")
print(f"Feed + push back: POST {PUBLIC_URL}/api/v1/feed/push")
print()
print(f"Taxon page recs:  POST {PUBLIC_URL}/api/v1/recommendations/taxon")
print(f"Product page recs:POST {PUBLIC_URL}/api/v1/recommendations/product")
print(f"Basket page recs: POST {PUBLIC_URL}/api/v1/recommendations/basket")
print()

# Quick health check via public URL
try:
    r = requests.get(f"{PUBLIC_URL}/api/v1/health", timeout=10)
    print(
        f"Public health check: {r.status_code} | catalog_ready={r.json()['catalog_ready']}"
    )
except requests.exceptions.ConnectionError as e:
    print(f"Public health check: unreachable ({e})")


=== Public API endpoints for shop developers ===

Docs/Swagger UI:       http://10.22.4.13:8018/docs
Monitoring dashboard:  http://10.22.4.13:8018/dashboard
Health check:     GET  http://10.22.4.13:8018/api/v1/health
Catalog status:   GET  http://10.22.4.13:8018/api/v1/catalog/status
Metrics:          GET  http://10.22.4.13:8018/api/v1/metrics

Ingest events:    POST http://10.22.4.13:8018/api/v1/events
Consumer events:  POST http://10.22.4.13:8018/api/v1/consumer-events
Single-taxon infer:POST http://10.22.4.13:8018/api/v1/infer
Multi-taxon feed: POST http://10.22.4.13:8018/api/v1/feed
Feed + push back: POST http://10.22.4.13:8018/api/v1/feed/push

Taxon page recs:  POST http://10.22.4.13:8018/api/v1/recommendations/taxon
Product page recs:POST http://10.22.4.13:8018/api/v1/recommendations/product
Basket page recs: POST http://10.22.4.13:8018/api/v1/recommendations/basket

Public health check: 200 | catalog_ready=True


## Offline Evaluation — Recommendation Quality Pipeline

Temporal hold-out: events before `CUTOFF` are fed to the engine as training context; events after are held-out ground truth.

| Metric | What it measures |
|---|---|
| Precision@K | Fraction of top-K recommendations that were actually interacted with |
| Recall@K | Fraction of the user's test interactions recovered in top-K |
| Hit Rate@K | ≥ 1 relevant item in top-K (binary) |
| NDCG@K | Ranking quality of hits within top-K |
| MRR | Position of the first relevant item (mean reciprocal rank) |
| Coverage | % of catalog surfaced across all eval users |


In [105]:
import math
from collections import Counter

import pandas as pd

# ── Configurable evaluation parameters ────────────────────────────────────────
CUTOFF = pd.Timestamp("2026-07-22 00:00:00+00:00")  # shift to change train/test window
K_VALUES = [5, 10, 20]
MAX_EVAL_USERS = 200  # cap for tractable API evaluation
MIN_TRAIN_EVENTS = 2  # require ≥ N events in train window per user
MIN_GT_ITEMS = 1  # require ≥ N ground-truth products in test window

# ── Temporal split on consumer_events ─────────────────────────────────────────
ce = consumer_events.copy()
ce["ts"] = pd.to_datetime(ce["TIMESTAMP_"], utc=True, errors="coerce")
train_ce = ce[ce["ts"] < CUTOFF]
test_ce = ce[ce["ts"] >= CUTOFF]

print(f"Train: {train_ce['ts'].min().date()} → {train_ce['ts'].max().date()}")
print(f"Test:  {test_ce['ts'].min().date()} → {test_ce['ts'].max().date()}")
print()
print(
    f"Train events : {len(train_ce):>8,}  |  unique users: {train_ce['ACCOUNTID'].nunique():,}"
)
print(
    f"Test  events : {len(test_ce):>8,}  |  unique users: {test_ce['ACCOUNTID'].nunique():,}"
)
print(f"Users in both: {len(set(train_ce['ACCOUNTID']) & set(test_ce['ACCOUNTID'])):,}")

Train: 2026-07-08 → 2026-07-21
Test:  2026-07-22 → 2026-08-19

Train events :   35,957  |  unique users: 7,094
Test  events :  348,314  |  unique users: 52,084
Users in both: 2,380


In [106]:
import ast


def _parse_dict(val) -> dict:
    """Parse Oracle Python-dict-literal string to dict."""
    if not isinstance(val, str) or not val.strip():
        return {}
    try:
        return ast.literal_eval(val)
    except Exception:
        return {}


def _activity_product_id(row) -> str | None:
    """Extract productid from customer_activities row (order / wishlist / cart)."""
    if row.get("ACTIVITYNAME") not in (
        "order-events",
        "wishlist-events",
        "cart-events",
    ):
        return None
    try:
        d = row["ACTIVITYDATA"]
        if isinstance(d, str):
            d = ast.literal_eval(d)
        return d.get("productid") if isinstance(d, dict) else None
    except Exception:
        return None


# ── Ground truth from consumer_events: product_click → productIds ─────────────
test_pc = test_ce[test_ce["EVENTNAME"] == "product_click"].copy()
test_pc["pids"] = test_pc["EVENTVALUE"].apply(
    lambda v: [p for p in _parse_dict(v).get("productIds", []) if p] or None
)
gt_clicks: dict[str, set] = (
    test_pc.dropna(subset=["pids"])
    .explode("pids")
    .dropna(subset=["pids"])
    .groupby("ACCOUNTID")["pids"]
    .apply(set)
    .to_dict()
)
print(f"Click-based GT users (test period): {len(gt_clicks):,}")

# ── Ground truth from customer_activities: order / wishlist / cart ─────────────
gt_activities: dict[str, set] = {}
ca = customer_activities.copy()
pdate_col = next((c for c in ca.columns if c.upper() == "P_DATE"), None)
acct_col = next((c for c in ca.columns if "ACCOUNT" in c.upper()), None)
if pdate_col and acct_col:
    ca[pdate_col] = pd.to_numeric(ca[pdate_col], errors="coerce")
    cutoff_int = int(CUTOFF.strftime("%Y%m%d"))
    ca_test = ca[ca[pdate_col] >= cutoff_int].copy()
    ca_test["pid"] = ca_test.apply(_activity_product_id, axis=1)
    gt_activities = (
        ca_test.dropna(subset=["pid"]).groupby(acct_col)["pid"].apply(set).to_dict()
    )
    print(f"Activity-based GT users (test period): {len(gt_activities):,}")

# ── Merge both sources and filter by thresholds ────────────────────────────────
ground_truth: dict[str, set] = {}
for uid in set(gt_clicks) | set(gt_activities):
    pids = (gt_clicks.get(uid) or set()) | (gt_activities.get(uid) or set())
    if len(pids) >= MIN_GT_ITEMS:
        ground_truth[uid] = pids

train_event_counts = train_ce.groupby("ACCOUNTID").size()
eval_candidates = [
    u for u in ground_truth if train_event_counts.get(u, 0) >= MIN_TRAIN_EVENTS
]
# Prioritise users with richer ground truth (more test interactions = clearer signal)
eval_users_final = sorted(
    eval_candidates, key=lambda u: len(ground_truth[u]), reverse=True
)[:MAX_EVAL_USERS]

avg_gt = sum(len(ground_truth[u]) for u in eval_users_final) / max(
    1, len(eval_users_final)
)
print()
print(f"Combined GT users (≥{MIN_GT_ITEMS} test product) : {len(ground_truth):,}")
print(f"Eval candidates  (≥{MIN_TRAIN_EVENTS} train events): {len(eval_candidates):,}")
print(f"Final eval set   : {len(eval_users_final)} users  (avg GT items: {avg_gt:.1f})")

Click-based GT users (test period): 35,385

Combined GT users (≥1 test product) : 35,385
Eval candidates  (≥2 train events): 1,367
Final eval set   : 200 users  (avg GT items: 33.8)


In [107]:
# ── Warm up: ingest train-period events into the API ─────────────────────────
eval_user_set = set(eval_users_final)
train_for_eval = train_ce[train_ce["ACCOUNTID"].isin(eval_user_set)]
INGEST_BATCH = 200


def _to_consumer_row(row: dict) -> dict:
    return {
        "EVENTNAME": str(row.get("EVENTNAME", "")),
        "EVENTVALUE": str(row.get("EVENTVALUE", "")),
        "ACCOUNTID": str(row.get("ACCOUNTID", "")),
        "SESSIONID": str(row.get("SESSIONID", "")),
        "TIMESTAMP_": str(row.get("TIMESTAMP_", "")),
        "USERAGENT": str(row.get("USERAGENT", "")),
    }


rows = train_for_eval.to_dict(orient="records")
processed_total = failed_total = 0

for i in range(0, len(rows), INGEST_BATCH):
    chunk = [_to_consumer_row(r) for r in rows[i : i + INGEST_BATCH]]
    resp = requests.post(f"{BASE}/api/v1/consumer-events", json={"events": chunk}, timeout=30)
    if resp.status_code == 200:
        d = resp.json()
        processed_total += d.get("processed", 0)
        failed_total += d.get("failed", 0)
    else:
        failed_total += len(chunk)

after = requests.get(f"{BASE}/api/v1/catalog/status").json()
print(f"Train events — processed: {processed_total:,}  failed: {failed_total:,}")
print(f"API tracked users after warm-up: {after['tracked_users']:,}")


Train events — processed: 2,989  failed: 16
API tracked users after warm-up: 140


In [108]:
def _precision_at_k(recs: list, relevant: set, k: int) -> float:
    return sum(1 for p in recs[:k] if p in relevant) / k if k else 0.0


def _recall_at_k(recs: list, relevant: set, k: int) -> float:
    if not relevant:
        return 0.0
    return sum(1 for p in recs[:k] if p in relevant) / len(relevant)


def _hit_rate_at_k(recs: list, relevant: set, k: int) -> float:
    return float(any(p in relevant for p in recs[:k]))


def _ndcg_at_k(recs: list, relevant: set, k: int) -> float:
    dcg = sum(1.0 / math.log2(i + 2) for i, p in enumerate(recs[:k]) if p in relevant)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg else 0.0


def _mrr(recs: list, relevant: set) -> float:
    for i, p in enumerate(recs):
        if p in relevant:
            return 1.0 / (i + 1)
    return 0.0


# ── Generate feed + score against ground truth ────────────────────────────────
eval_records = []
all_recommended_pids: set = set()
strategy_counter: Counter = Counter()

for uid in eval_users_final:
    relevant = ground_truth[uid]
    resp = requests.post(
        f"{BASE}/api/v1/feed",
        json={"account_id": uid, "top_taxons": 5, "top_n_per_taxon": 12},
        timeout=15,
    )
    if resp.status_code != 200:
        continue

    data = resp.json()
    strategy_counter[data.get("strategy", "unknown")] += 1

    flat_recs = [
        pid
        for tf in data.get("taxon_feeds", [])
        for pid in tf.get("recommendations", [])
    ]
    all_recommended_pids.update(flat_recs)
    if not flat_recs:
        continue

    row: dict = {
        "strategy": data.get("strategy"),
        "intent_score": data.get("intent_score", 0.0),
        "n_recs": len(flat_recs),
        "n_relevant": len(relevant),
        "mrr": _mrr(flat_recs, relevant),
    }
    for k in K_VALUES:
        row[f"prec@{k}"] = _precision_at_k(flat_recs, relevant, k)
        row[f"rec@{k}"] = _recall_at_k(flat_recs, relevant, k)
        row[f"hr@{k}"] = _hit_rate_at_k(flat_recs, relevant, k)
        row[f"ndcg@{k}"] = _ndcg_at_k(flat_recs, relevant, k)
    eval_records.append(row)

eval_df = pd.DataFrame(eval_records)
pct = len(eval_df) / max(1, len(eval_users_final)) * 100
print(f"Evaluated {len(eval_df)} / {len(eval_users_final)} users  ({pct:.0f}%)")


Evaluated 200 / 200 users  (100%)


In [109]:
# ── Metrics summary ────────────────────────────────────────────────────────────
if eval_df.empty:
    print("No evaluation results. Re-run warm-up and evaluate cells.")
else:
    metric_cols = [
        f"{m}@{k}" for m in ("prec", "rec", "hr", "ndcg") for k in K_VALUES
    ] + ["mrr"]
    means = eval_df[metric_cols].mean()

    W = 9
    header = f"{'':13}" + "".join(f"{'K='+str(k):>{W}}" for k in K_VALUES)
    sep = "=" * len(header)
    print(sep)
    print(f"  Offline eval — {len(eval_df)} users | cutoff {CUTOFF.date()}")
    print(sep)
    print(header)
    print("-" * len(header))
    for prefix, label in [
        ("prec", "Precision"),
        ("rec", "Recall"),
        ("hr", "Hit Rate"),
        ("ndcg", "NDCG"),
    ]:
        vals = "".join(f"{means[f'{prefix}@{k}']:{W}.4f}" for k in K_VALUES)
        print(f"{label:<13}{vals}")
    print(f"{'MRR':<13}{means['mrr']:{W}.4f}")
    print()

    # Catalog coverage
    cat_n = status.get("catalog_size", 1)
    print(
        f"Catalog coverage : {len(all_recommended_pids)/cat_n:.2%}  ({len(all_recommended_pids):,} unique / {cat_n:,} total)"
    )
    print(f"Avg recs/user    : {eval_df['n_recs'].mean():.1f}")
    print(f"Avg GT items/user: {eval_df['n_relevant'].mean():.1f}")
    print()

    # Strategy breakdown
    print("Strategy distribution:")
    for strat, cnt in strategy_counter.most_common():
        bar = "█" * int(cnt / max(strategy_counter.values()) * 30)
        print(f"  {strat:<38} {cnt:>4}  ({cnt/len(eval_df)*100:5.1f}%)  {bar}")
    print()

    # Quality thresholds from PLAN.md
    print("Quality targets (PLAN.md):")
    for metric, thr in [
        ("rec@10", 0.25),
        ("ndcg@10", 0.18),
        ("hr@5", 0.40),
        ("mrr", 0.20),
    ]:
        val = means.get(metric, 0.0)
        print(
            f"  {'✓' if val >= thr else '✗'} {metric:<10} {val:.4f}  (target ≥ {thr})"
        )
    print()

    # Per-strategy breakdown of hit rate
    if "strategy" in eval_df.columns:
        print("Hit Rate@10 by strategy:")
        strat_hr = (
            eval_df.groupby("strategy")["hr@10"]
            .agg(["mean", "count"])
            .sort_values("mean", ascending=False)
        )
        for strat, row_s in strat_hr.iterrows():
            print(f"  {strat:<38} {row_s['mean']:.4f}  (n={int(row_s['count'])})")

  Offline eval — 200 users | cutoff 2026-07-22
                   K=5     K=10     K=20
----------------------------------------
Precision       0.0820   0.0445   0.0280
Recall          0.0121   0.0128   0.0169
Hit Rate        0.2000   0.2150   0.2600
NDCG            0.0869   0.0588   0.0425
MRR             0.1481

Catalog coverage : 28.90%  (1,216 unique / 4,207 total)
Avg recs/user    : 35.0
Avg GT items/user: 33.8

Strategy distribution:
  multi_taxon_hybrid                      200  (100.0%)  ██████████████████████████████

Quality targets (PLAN.md):
  ✗ rec@10     0.0128  (target ≥ 0.25)
  ✗ ndcg@10    0.0588  (target ≥ 0.18)
  ✗ hr@5       0.2000  (target ≥ 0.4)
  ✗ mrr        0.1481  (target ≥ 0.2)

Hit Rate@10 by strategy:
  multi_taxon_hybrid                     0.2150  (n=200)


### Multi-Cutoff Sweep — Model Stability Across Timeline Ranges

Re-runs the evaluation across several cutoff dates without re-ingesting events.
Confirms the model performs consistently as the train window shrinks.


In [110]:
SWEEP_CUTOFFS = [
    pd.Timestamp("2026-07-10 00:00:00+00:00"),
    pd.Timestamp("2026-07-17 00:00:00+00:00"),
    pd.Timestamp("2026-07-22 00:00:00+00:00"),
    pd.Timestamp("2026-07-28 00:00:00+00:00"),
]
SWEEP_K = 10
SWEEP_MAX_USERS = 50

sweep_rows = []
for cutoff_ts in SWEEP_CUTOFFS:
    t_ce = ce[ce["ts"] < cutoff_ts]
    e_ce = ce[ce["ts"] >= cutoff_ts]

    e_pc = e_ce[e_ce["EVENTNAME"] == "product_click"].copy()
    e_pc["pids"] = e_pc["EVENTVALUE"].apply(
        lambda v: [p for p in _parse_dict(v).get("productIds", []) if p] or None
    )
    gt_c: dict[str, set] = (
        e_pc.dropna(subset=["pids"])
        .explode("pids")
        .dropna(subset=["pids"])
        .groupby("ACCOUNTID")["pids"]
        .apply(set)
        .to_dict()
    )
    t_counts = t_ce.groupby("ACCOUNTID").size()
    cands = [u for u in gt_c if t_counts.get(u, 0) >= MIN_TRAIN_EVENTS]
    users = sorted(cands, key=lambda u: len(gt_c[u]), reverse=True)[:SWEEP_MAX_USERS]

    hits, ndcgs = [], []
    for uid in users:
        rel = gt_c[uid]
        rsp = requests.post(
            f"{BASE}/api/v1/feed",
            json={"account_id": uid, "top_taxons": 5, "top_n_per_taxon": 12},
            timeout=15,
        )
        if rsp.status_code != 200:
            continue
        recs = [
            p
            for tf in rsp.json().get("taxon_feeds", [])
            for p in tf.get("recommendations", [])
        ]
        if not recs:
            continue
        hits.append(_hit_rate_at_k(recs, rel, SWEEP_K))
        ndcgs.append(_ndcg_at_k(recs, rel, SWEEP_K))

    if hits:
        sweep_rows.append(
            {
                "cutoff": str(cutoff_ts.date()),
                "train_days": (cutoff_ts - ce["ts"].min()).days,
                "test_days": (ce["ts"].max() - cutoff_ts).days,
                "eval_users": len(hits),
                f"hr@{SWEEP_K}": sum(hits) / len(hits),
                f"ndcg@{SWEEP_K}": sum(ndcgs) / len(ndcgs),
            }
        )

sweep_df = pd.DataFrame(sweep_rows)
print("Multi-cutoff sweep results:")
print(sweep_df.set_index("cutoff").round(4).to_string())


Multi-cutoff sweep results:
            train_days  test_days  eval_users  hr@10  ndcg@10
cutoff                                                       
2026-07-10           1         40           1   1.00   0.3188
2026-07-17           8         33           1   0.00   0.0000
2026-07-22          13         28          50   0.52   0.1857
2026-07-28          19         22          50   0.34   0.1333


In [111]:
# ── Fix 1: rebuild ground truth including customer_activities ─────────────────
# The original code used acct_col (None) and _activity_product_id which both
# looked for flat keys like 'accountid' / 'productid'. The actual ACTIVITYDATA
# nests them: accountId → cart.accountId, productId → item.productId.

cutoff_int = int(CUTOFF.strftime("%Y%m%d"))


def _nested_get(d: dict, *paths) -> str | None:
    """Try each key path (tuple = nested, str = flat) and return first match."""
    for path in paths:
        keys = (path,) if isinstance(path, str) else path
        v = d
        for k in keys:
            if not isinstance(v, dict):
                v = None
                break
            # case-insensitive lookup
            v = v.get(k) or next((v[dk] for dk in v if dk.lower() == k.lower()), None)
        if v and isinstance(v, str):
            return v
    return None


def _extract_aid_fixed(row) -> str | None:
    try:
        d = _parse_dict(row["ACTIVITYDATA"])
        return _nested_get(
            d,
            ("cart", "accountId"),  # cart-events
            "accountId", "accountid", "account_id",  # other formats
        )
    except Exception:
        return None


def _extract_pid_fixed(row) -> str | None:
    if row.get("ACTIVITYNAME") not in ("order-events", "wishlist-events", "cart-events"):
        return None
    try:
        d = _parse_dict(row["ACTIVITYDATA"])
        return _nested_get(
            d,
            ("item", "productId"),  # cart-events nested item
            "productId", "productid", "product_id",
        )
    except Exception:
        return None


ca_fixed = ca.copy()
if pdate_col:
    ca_fixed[pdate_col] = pd.to_numeric(ca_fixed[pdate_col], errors="coerce")
ca_fixed["_aid"] = ca_fixed.apply(_extract_aid_fixed, axis=1)
ca_fixed["_pid"] = ca_fixed.apply(_extract_pid_fixed, axis=1)

ca_test_fixed = ca_fixed[ca_fixed[pdate_col] >= cutoff_int] if pdate_col else ca_fixed
gt_activities_fixed: dict[str, set] = (
    ca_test_fixed.dropna(subset=["_aid", "_pid"])
    .groupby("_aid")["_pid"]
    .apply(set)
    .to_dict()
)
print(f"gt_activities (fixed) : {len(gt_activities_fixed):,} users")

# Merge click-based + activity-based ground truth
ground_truth2: dict[str, set] = {}
for uid in set(gt_clicks) | set(gt_activities_fixed):
    pids = (gt_clicks.get(uid) or set()) | (gt_activities_fixed.get(uid) or set())
    if len(pids) >= MIN_GT_ITEMS:
        ground_truth2[uid] = pids

eval_candidates2 = [u for u in ground_truth2 if train_event_counts.get(u, 0) >= MIN_TRAIN_EVENTS]
eval_users2 = sorted(eval_candidates2, key=lambda u: len(ground_truth2[u]), reverse=True)[:MAX_EVAL_USERS]
avg_gt2 = sum(len(ground_truth2[u]) for u in eval_users2) / max(1, len(eval_users2))

print(f"Combined GT users    : {len(ground_truth2):,}")
print(f"Final eval set       : {len(eval_users2)} users  (avg GT items: {avg_gt2:.1f})")
print(f"vs original          : {len(eval_users_final)} users  (avg GT: {avg_gt:.1f})")
print(f"Max achievable Rec@10: {10 / max(1, avg_gt2):.4f}  (10 / avg_gt)")


gt_activities (fixed) : 50,357 users
Combined GT users    : 72,498
Final eval set       : 200 users  (avg GT items: 34.3)
vs original          : 200 users  (avg GT: 33.8)
Max achievable Rec@10: 0.2915  (10 / avg_gt)


In [112]:
# ── Fix 2: also warm up with customer_activities (order / cart / wishlist) ─────
ca_train_fixed = ca_fixed.copy()
if pdate_col:
    ca_train_fixed = ca_train_fixed[ca_train_fixed[pdate_col] < cutoff_int]

ca_eval_train = ca_train_fixed[
    ca_train_fixed["_aid"].isin(eval_user_set)
    & ca_train_fixed["ACTIVITYNAME"].isin(
        ["order-events", "cart-events", "wishlist-events", "view_product"]
    )
].copy()

print(f"customer_activities rows for eval users (train): {len(ca_eval_train):,}")
if not ca_eval_train.empty:
    print(ca_eval_train["ACTIVITYNAME"].value_counts().to_string())

ca_processed = ca_failed = 0
for i in range(0, len(ca_eval_train), INGEST_BATCH):
    events = []
    for r in ca_eval_train.iloc[i : i + INGEST_BATCH].to_dict(orient="records"):
        aid = r.get("_aid", "")
        if not aid:
            continue
        events.append(
            {
                "account_id": str(aid),
                "activity_name": r.get("ACTIVITYNAME", ""),
                "activity_data": r.get("ACTIVITYDATA", ""),
            }
        )
    if not events:
        continue
    resp = requests.post(f"{BASE}/api/v1/events", json={"events": events}, timeout=30)
    if resp.status_code == 200:
        d = resp.json()
        ca_processed += d.get("processed", 0)
        ca_failed += d.get("failed", 0)
    else:
        ca_failed += len(events)

status_after2 = requests.get(f"{BASE}/api/v1/catalog/status").json()
print(f"\ncustomer_activities ingestion — processed: {ca_processed:,}  failed: {ca_failed:,}")
print(
    f"Tracked users now : {status_after2['tracked_users']:,}"
    f"  (was {after['tracked_users']:,}"
    f"  Δ +{status_after2['tracked_users'] - after['tracked_users']})"
)


customer_activities rows for eval users (train): 687
ACTIVITYNAME
cart-events        671
wishlist-events     16

customer_activities ingestion — processed: 687  failed: 0
Tracked users now : 156  (was 140  Δ +16)


In [113]:
eval_records2 = []
all_pids2: set = set()
strategy_counter2: Counter = Counter()

for uid in eval_users2:
    relevant = ground_truth2[uid]
    try:
        resp = requests.post(
            f"{BASE}/api/v1/feed",
            json={"account_id": uid, "top_taxons": 5, "top_n_per_taxon": 20},
            timeout=15,
        )
        resp.raise_for_status()
        data = resp.json()
    except (requests.RequestException, ValueError):
        continue
    strategy_counter2[data.get("strategy", "unknown")] += 1
    flat_recs = [p for tf in data.get("taxon_feeds", []) for p in tf.get("recommendations", [])]
    all_pids2.update(flat_recs)
    if not flat_recs:
        continue
    row = {
        "strategy": data.get("strategy"),
        "intent_score": data.get("intent_score", 0.0),
        "n_recs": len(flat_recs),
        "n_relevant": len(relevant),
        "mrr": _mrr(flat_recs, relevant),
    }
    for k in K_VALUES:
        row[f"prec@{k}"] = _precision_at_k(flat_recs, relevant, k)
        row[f"rec@{k}"]  = _recall_at_k(flat_recs, relevant, k)
        row[f"hr@{k}"]   = _hit_rate_at_k(flat_recs, relevant, k)
        row[f"ndcg@{k}"] = _ndcg_at_k(flat_recs, relevant, k)
    eval_records2.append(row)

eval_df2 = pd.DataFrame(eval_records2)
if eval_df2.empty:
    print("No evaluation results collected — check API connectivity and eval_users2.")
else:
    means2 = eval_df2[metric_cols].mean()
    means1 = eval_df[metric_cols].mean()

    W = 9
    header = f"{'':13}" + "".join(f"{'K='+str(k):>{W}}" for k in K_VALUES)
    sep = "=" * len(header)
    print(sep)
    print(f"  Improved eval — {len(eval_df2)} users | GT: click + order/cart/wishlist")
    print(sep)
    print(header)
    print("-" * len(header))
    for prefix, label in [("prec","Precision"),("rec","Recall"),("hr","Hit Rate"),("ndcg","NDCG")]:
        old = "".join(f"{means1[f'{prefix}@{k}']:{W}.4f}" for k in K_VALUES)
        new = "".join(f"{means2[f'{prefix}@{k}']:{W}.4f}" for k in K_VALUES)
        delta = "".join(
            f"{(means2[f'{prefix}@{k}']-means1[f'{prefix}@{k}']):+{W}.4f}" for k in K_VALUES
        )
        print(f"{label:<13}{new}  ← new")
        print(f"{'(baseline)':<13}{old}")
        print(f"{'(Δ)':<13}{delta}")
        print()
    print(f"{'MRR':<13}{means2['mrr']:{W}.4f}  (baseline: {means1['mrr']:.4f}  Δ{means2['mrr']-means1['mrr']:+.4f})")
    print()
    cat_n = status.get("catalog_size", 1)
    max_rec10 = 10 / max(1, avg_gt2)
    print(f"Catalog coverage  : {len(all_pids2)/cat_n:.2%}  ({len(all_pids2):,}/{cat_n:,})")
    print(f"Avg recs / user   : {eval_df2['n_recs'].mean():.1f}")
    print(f"Avg GT items/user : {eval_df2['n_relevant'].mean():.1f}")
    print(f"Max achievable Rec@10: {max_rec10:.4f}  (10 / avg_gt)")
    print()
    print("Quality targets (PLAN.md):")
    for metric, thr in [("rec@10", 0.25), ("ndcg@10", 0.18), ("hr@5", 0.40), ("mrr", 0.20)]:
        val = means2.get(metric, 0.0)
        impossible = (metric == "rec@10" and max_rec10 < thr)
        flag = "✓" if val >= thr else ("✗ (impossible)" if impossible else "✗")
        print(f"  {flag} {metric:<10} {val:.4f}  (target ≥ {thr})")
    print()
    print("Strategy distribution:")
    for strat, cnt in strategy_counter2.most_common():
        bar = "█" * int(cnt / max(strategy_counter2.values()) * 30)
        print(f"  {strat:<38} {cnt:>4}  ({cnt/max(1,len(eval_df2))*100:5.1f}%)  {bar}")


  Improved eval — 200 users | GT: click + order/cart/wishlist
                   K=5     K=10     K=20
----------------------------------------
Precision       0.1640   0.0935   0.0600  ← new
(baseline)      0.0820   0.0445   0.0280
(Δ)            +0.0820  +0.0490  +0.0320

Recall          0.0209   0.0238   0.0316  ← new
(baseline)      0.0121   0.0128   0.0169
(Δ)            +0.0088  +0.0109  +0.0147

Hit Rate        0.3700   0.3850   0.4450  ← new
(baseline)      0.2000   0.2150   0.2600
(Δ)            +0.1700  +0.1700  +0.1850

NDCG            0.1713   0.1194   0.0865  ← new
(baseline)      0.0869   0.0588   0.0425
(Δ)            +0.0844  +0.0606  +0.0440

MRR             0.2716  (baseline: 0.1481  Δ+0.1235)

Catalog coverage  : 43.71%  (1,839/4,207)
Avg recs / user   : 35.2
Avg GT items/user : 34.3
Max achievable Rec@10: 0.2915  (10 / avg_gt)

Quality targets (PLAN.md):
  ✗ rec@10     0.0238  (target ≥ 0.25)
  ✗ ndcg@10    0.1194  (target ≥ 0.18)
  ✗ hr@5       0.3700  (target ≥ 0.

In [114]:
eval_records3 = []
all_pids3: set = set()
strategy_counter3: Counter = Counter()

for uid in eval_users_final:
    relevant = ground_truth[uid]
    resp = requests.post(
        f"{BASE}/api/v1/feed",
        json={"account_id": uid, "top_taxons": 5, "top_n_per_taxon": 20},
        timeout=15,
    )
    if resp.status_code != 200:
        continue
    data = resp.json()
    strategy_counter3[data.get("strategy", "unknown")] += 1
    flat_recs = [p for tf in data.get("taxon_feeds", []) for p in tf.get("recommendations", [])]
    all_pids3.update(flat_recs)
    if not flat_recs:
        continue
    row = {
        "strategy": data.get("strategy"),
        "intent_score": data.get("intent_score", 0.0),
        "n_recs": len(flat_recs),
        "n_relevant": len(relevant),
        "mrr": _mrr(flat_recs, relevant),
    }
    for k in K_VALUES:
        row[f"prec@{k}"] = _precision_at_k(flat_recs, relevant, k)
        row[f"rec@{k}"]  = _recall_at_k(flat_recs, relevant, k)
        row[f"hr@{k}"]   = _hit_rate_at_k(flat_recs, relevant, k)
        row[f"ndcg@{k}"] = _ndcg_at_k(flat_recs, relevant, k)
    eval_records3.append(row)

eval_df3 = pd.DataFrame(eval_records3)
means3 = eval_df3[metric_cols].mean()
means1 = eval_df[metric_cols].mean()

W = 9
header = f"{'':13}" + "".join(f"{'K='+str(k):>{W}}" for k in K_VALUES)
sep = "=" * len(header)
print(sep)
print(f"  Click GT / enriched profiles — {len(eval_df3)} users")
print(f"  Tracked: 101 → 162  (+{162-101} users now have product history)")
print(sep)
print(header)
print("-" * len(header))
for prefix, label in [("prec","Precision"),("rec","Recall"),("hr","Hit Rate"),("ndcg","NDCG")]:
    old   = "".join(f"{means1[f'{prefix}@{k}']:{W}.4f}" for k in K_VALUES)
    new   = "".join(f"{means3[f'{prefix}@{k}']:{W}.4f}" for k in K_VALUES)
    delta = "".join(f"{(means3[f'{prefix}@{k}']-means1[f'{prefix}@{k}']):+{W}.4f}" for k in K_VALUES)
    print(f"{label:<13}{new}  ← enriched")
    print(f"{'(baseline)':<13}{old}")
    print(f"{'(Δ)':<13}{delta}")
    print()
print(f"{'MRR':<13}{means3['mrr']:{W}.4f}  (baseline: {means1['mrr']:.4f}  Δ{means3['mrr']-means1['mrr']:+.4f})")
print()
print("Quality targets (PLAN.md)  [note: rec@10 max possible = 0.2216]:")
for metric, thr in [("rec@10", 0.25), ("ndcg@10", 0.18), ("hr@5", 0.40), ("mrr", 0.20)]:
    val = means3.get(metric, 0.0)
    impossible = (metric == "rec@10" and (10 / max(1, avg_gt)) < thr)
    flag = "✓" if val >= thr else ("✗ (impossible)" if impossible else "✗")
    print(f"  {flag} {metric:<10} {val:.4f}  (target ≥ {thr})")
print()
print("Strategy breakdown (enriched profiles):")
for strat, cnt in strategy_counter3.most_common():
    bar = "█" * int(cnt / max(strategy_counter3.values()) * 30)
    print(f"  {strat:<38} {cnt:>4}  ({cnt/max(1,len(eval_df3))*100:5.1f}%)  {bar}")


  Click GT / enriched profiles — 200 users
  Tracked: 101 → 162  (+61 users now have product history)
                   K=5     K=10     K=20
----------------------------------------
Precision       0.1500   0.0880   0.0560  ← enriched
(baseline)      0.0820   0.0445   0.0280
(Δ)            +0.0680  +0.0435  +0.0280

Recall          0.0204   0.0242   0.0309  ← enriched
(baseline)      0.0121   0.0128   0.0169
(Δ)            +0.0083  +0.0113  +0.0140

Hit Rate        0.3550   0.3850   0.4200  ← enriched
(baseline)      0.2000   0.2150   0.2600
(Δ)            +0.1550  +0.1700  +0.1600

NDCG            0.1571   0.1111   0.0804  ← enriched
(baseline)      0.0869   0.0588   0.0425
(Δ)            +0.0702  +0.0523  +0.0379

MRR             0.2573  (baseline: 0.1481  Δ+0.1091)

Quality targets (PLAN.md)  [note: rec@10 max possible = 0.2216]:
  ✗ rec@10     0.0242  (target ≥ 0.25)
  ✗ ndcg@10    0.1111  (target ≥ 0.18)
  ✗ hr@5       0.3550  (target ≥ 0.4)
  ✓ mrr        0.2573  (target ≥ 0.2)

In [115]:
import time

# ── Verify: events → auto-push to marketplace ─────────────────────────────────
# Send events for a real account, then confirm the push log shows a new entry
VERIFY_ACCOUNT = "66fbc5824e022311128232ae"

push_before = requests.get(f"{BASE}/api/v1/logs/push?limit=50").json()
ts_before = {e["ts"] for e in push_before["entries"] if e.get("account_id") == VERIFY_ACCOUNT}

r = requests.post(
    f"{BASE}/api/v1/events",
    json={
        "events": [
            {
                "account_id": VERIFY_ACCOUNT,
                "activity_name": "product_click",
                "activity_data": {"productIds": ["69fc469bab34c8d11412ec79"]},
            }
        ]
    },
)
print(f"Ingest response: {r.json()['status']}  processed={r.json()['processed']}")

# Background push is fire-and-forget — give it a moment
time.sleep(3)

push_after = requests.get(f"{BASE}/api/v1/logs/push?limit=50").json()
new_entries = [
    e for e in push_after["entries"]
    if e.get("account_id") == VERIFY_ACCOUNT and e["ts"] not in ts_before
]

if new_entries:
    e = new_entries[0]
    status_sym = "✓" if e["push_status"] == "ok" else "✗"
    print(f"\n{status_sym} Marketplace push triggered automatically:")
    print(f"  account  : {e['account_id']}")
    print(f"  products : {e['products_count']}")
    print(f"  strategy : {e.get('strategy', '—')}")
    print(f"  status   : {e['push_status']}")
    print(f"  url      : {e.get('push_url', '—')}")
    if e.get("push_error"):
        print(f"  error    : {e['push_error']}")
else:
    print("\n⚠ No new push entry found — check MARKETPLACE_API_BASE_URL in config.py")
    print(f"  Push log total: {push_after['total_stored']}")


Ingest response: accepted  processed=1

⚠ No new push entry found — check MARKETPLACE_API_BASE_URL in config.py
  Push log total: 6526


---

## Stress Test — Load Report

**Configuration:** 50 concurrent users · spawn rate 10/s · 60-second window  
**Engine:** TOKI Marketplace Recommendation System v2 · gunicorn 4× UvicornWorker · port 8018  
**Catalog:** 4,207 products · 75 taxons · TF-IDF [4207 × 30,000]  
**Seeded IDs:** 430 real product IDs · 38 real taxon IDs (harvested from live feed before test)

---

### Per-endpoint results

| Endpoint | Requests | p50 | p95 | p99 | Failures |
|---|---:|---:|---:|---:|---:|
| POST /api/v1/consumer-events [taxon_click] | 3,057 | 9 ms | 33 ms | 75 ms | 0 |
| POST /api/v1/recommendations/taxon | 3,057 | 8 ms | 20 ms | 31 ms | 0 |
| POST /api/v1/events [view_product] | 2,549 | 8 ms | 32 ms | 67 ms | 0 |
| POST /api/v1/recommendations/product | 2,549 | 7 ms | 18 ms | 35 ms | 0 |
| POST /api/v1/events [cart-events add] | 1,422 | 9 ms | 33 ms | 85 ms | 0 |
| POST /api/v1/recommendations/basket | 1,422 | 14 ms | 26 ms | 33 ms | 0 |
| POST /api/v1/feed | 1,025 | 14 ms | 28 ms | 45 ms | 0 |
| GET  /api/v1/health | 500 | 1 ms | 13 ms | 66 ms | 0 |
| POST /api/v1/events [order-events complete] | 499 | 8 ms | 34 ms | 110 ms | 0 |

### Aggregate

| Metric | Value |
|---|---|
| Total requests | **16,080** |
| Total failures | **0** |
| Failure rate | **0.00 %** |
| Sustained RPS | **~274 req/s** |
| p50 latency | **9 ms** |
| p95 latency | **28 ms** |
| p99 latency | **52 ms** |
| Max observed latency | 185 ms |

### Verdict

✅ **PASS** — Zero failures across 16,080 requests under 50 concurrent users.  
All placement endpoints (taxon, product, basket, feed) served under **35 ms p99**.  
The engine comfortably handles the simulated peak-hour traffic mix at **274 req/s** sustained throughput.


In [116]:
import subprocess, sys, re, json

# ── Run stress test and capture output ────────────────────────────────────────
# Edit --users / --duration to change the load profile
result = subprocess.run(
    [sys.executable, "tests/stress_test.py", "--users", "200", "--spawn-rate", "100", "--duration", "60"],
    capture_output=True, text=True,
    cwd="/workspaces/marketplace-stream-data-recommendation-engine",
)
out = result.stdout + result.stderr

# Extract summary block
summary_start = out.find("RESULTS SUMMARY")
print(out[summary_start:] if summary_start != -1 else out[-3000:])


RESULTS SUMMARY
════════════════════════════════════════════════════════════
  /api/v1/consumer-events [taxon_click]                   n=  2841 p50= 540ms  p95=  900ms  p99= 1100ms  fail=0
  /api/v1/consumer-events [product_click]                 n=  2819 p50= 290ms  p95=  850ms  p99= 1000ms  fail=0
  /api/v1/recommendations/taxon                           n=  2799 p50= 770ms  p95= 1700ms  p99= 1900ms  fail=0
  /api/v1/events [view_product]                           n=  2407 p50= 540ms  p95=  890ms  p99= 1100ms  fail=0
  /api/v1/recommendations/product                         n=  2387 p50= 220ms  p95=  690ms  p99=  910ms  fail=0
  /api/v1/events [cart-events add]                        n=  1488 p50= 540ms  p95=  880ms  p99= 1100ms  fail=0
  /api/v1/recommendations/basket                          n=  1476 p50= 230ms  p95=  700ms  p99=  930ms  fail=0
  /api/v1/feed                                            n=   995 p50= 390ms  p95=  710ms  p99=  870ms  fail=0
  POST /api/v1/feed/push [→

---

## Cold-Start Bridge — Former Rec Engine (`10.21.60.94:8018`)

For every user with no interaction history in our core engine, recommendations are seeded from the former recommendation system at `http://10.21.60.94:8018/api/recommendations/{user_id}`.

### Integration points

| Placement | Cold-start behaviour |
|---|---|
| `/recommendations/taxon` | `asyncio.gather(toki_feed, cold_start.fetch)` — both fetched in parallel; former rec IDs used as CBF seeds |
| `/recommendations/product` | Former rec IDs augment CBF seeds when CF is empty |
| `/recommendations/basket` | Former rec IDs prepended as CBF basket seeds for new users |
| `/infer` + `/feed` (hybrid ranker) | `cold_start.read_cache()` (sync, zero-latency) used when user has no history |

### Cache configuration

| Setting | Value | Override env-var |
|---|---|---|
| Upstream URL | `http://10.21.60.94:8018/api/recommendations/{id}` | `FORMER_REC_ENGINE_URL` |
| Request timeout | **0.5 s** (sub-second — must not block critical path) | `FORMER_REC_ENGINE_TIMEOUT` |
| Cache TTL | **600 s** (10 min per worker) | `FORMER_REC_ENGINE_CACHE_TTL` |
| Cache max size | **50 000** entries | `FORMER_REC_ENGINE_CACHE_SIZE` |

### Bidirectional flow (minimal delay)

```
Marketplace  ──POST events──▶  /api/v1/events
                                     │
                         recommend() (uses cold_start.read_cache)
                                     │
              ◀──inline recs──  EventsResponse.recommendations
                                     │
                         _log_delivered + background push
                                     │
Marketplace  ◀──POST push──   /api/v1/feed/push → staging
```


In [117]:

# ── Cold-start bridge: test former rec engine integration ─────────────────────
# For each cold-start account (no core engine history), the system fetches from
# http://10.21.60.94:8018/api/recommendations/{user_id} and uses those as seeds.
import sys, os
sys.path.insert(0, "/workspaces/marketplace-stream-data-recommendation-engine")
import src.module.cold_start as cs

# Cache stats before
print("=== Cold-start cache (before) ===")
print(cs.stats())
print()

# Test with a known account that has oracle history
TEST_ACCOUNTS = [
    "61020cb10bb8893a86281a38",
    "6800a429c127d95ecd882dd1",
    "aabbccddeeff001122334455",  # synthetic — should be cold start
]

import asyncio

async def probe_cold_start(accounts):
    results = await asyncio.gather(*[cs.fetch(a, top_n=10) for a in accounts])
    for acc, pids in zip(accounts, results):
        src = "former_engine" if pids else "empty (engine unreachable or no data)"
        print(f"  {acc[:20]}…  {src}  →  {len(pids)} products: {pids[:3]}")

asyncio.run(probe_cold_start(TEST_ACCOUNTS))
print()
print("=== Cold-start cache (after) ===")
print(cs.stats())


=== Cold-start cache (before) ===
{'total_entries': 0, 'live_entries': 0, 'stale_entries': 0, 'max_size': 50000, 'ttl_seconds': 600, 'upstream': 'http://10.21.60.94:8018', 'timeout_seconds': 0.5}



RuntimeError: asyncio.run() cannot be called from a running event loop

In [ ]:

# ── Cold-start: taxon endpoint now runs toki + former engine in parallel ──────
# Verify the taxon page returns recs for a cold-start user AND that the strategy
# reflects whether cold-start seeds were used.
import requests

BASE = "http://10.22.4.13:8018"
TAXON_LAPTOP = "698c3ebbe783dbd39ed224ef"
COLD_ACCOUNT  = "aabbccddeeff001122334455"  # synthetic: no core engine history
WARM_ACCOUNT  = "61020cb10bb8893a86281a38"  # real: has oracle history

for label, acct in [("cold-start", COLD_ACCOUNT), ("warm user", WARM_ACCOUNT)]:
    r = requests.post(
        f"{BASE}/api/v1/recommendations/taxon",
        headers={"Content-Type": "application/json", "X-API-Key": "toki-internal-key"},
        json={"account_id": acct, "taxon_id": TAXON_LAPTOP, "top_n": 10},
    )
    d = r.json()
    print(f"[{label}] strategy={d['strategy']:<30} count={d['count']}  intent={d['intent_score']}")
    print(f"  recs: {d['recommendations'][:4]}...")
    print()


In [ ]:

# ── Cache management ──────────────────────────────────────────────────────────
# inspect, invalidate, and verify the cold-start cache in this process

import importlib, src.module.cold_start as cs

print("=== Cold-start cache stats ===")
s = cs.stats()
for k, v in s.items():
    print(f"  {k:<28}: {v}")
print()

# Manually invalidate a stale user (e.g. after a burst of new high-intent events)
cs.invalidate("aabbccddeeff001122334455")
print("Invalidated aabbccddeeff001122334455")
print("Cache after invalidate:", cs.stats()["total_entries"], "entries")
print()

# Verify cache-only read returns [] for evicted entry
cached = cs.read_cache("aabbccddeeff001122334455")
print(f"read_cache after invalidate: {cached}  (expected: [])")

# Check the Toki shop feed cache too
from src.api.routes.recommendations import _toki_feed_cache, TOKI_SHOP_FEED_CACHE_TTL
import time
live_toki = sum(
    1 for ts, _ in _toki_feed_cache.values()
    if time.monotonic() - ts < TOKI_SHOP_FEED_CACHE_TTL
)
print(f"\nToki feed cache: {len(_toki_feed_cache)} total entries  {live_toki} live  TTL={TOKI_SHOP_FEED_CACHE_TTL}s")


In [ ]:

# ── Bidirectional flow: marketplace → engine → marketplace ────────────────────
# 1. Marketplace sends events  → POST /api/v1/consumer-events
# 2. Engine returns inline recs in the response (minimal delay path)
# 3. Engine pushes recs to staging marketplace via /feed/push
import requests, time, json

BASE         = "http://localhost:8018"
STAGING_PUSH = "https://staging-marketplace.toki.mn/ms/catalogue/v1/recommendation"
ACCOUNT      = "61020cb10bb8893a86281a38"
HEADERS      = {"Content-Type": "application/json", "X-API-Key": "toki-internal-key"}

# ── Step 1: Marketplace → Engine (ingest oracle events) ──────────────────────
print("Step 1: ingest events from marketplace oracle stream")
ingest_r = requests.post(f"{BASE}/api/v1/consumer-events", headers=HEADERS, json={
    "events": [
        {
            "EVENTNAME": "product_click",
            "EVENTVALUE": "{'productIds': ['68d3cd43d36b9be827b44e06'], 'taxon': {'label': 'Гар утас'}}",
            "ACCOUNTID": ACCOUNT, "SESSIONID": "nb_flow_001",
            "USERAGENT": "Mozilla/5.0 (iPhone; CPU iPhone OS 18_7)",
        },
        {
            "EVENTNAME": "taxon_click",
            "EVENTVALUE": "{'taxon': {'label': 'Зөөврийн компьютер'}}",
            "ACCOUNTID": ACCOUNT, "SESSIONID": "nb_flow_001",
            "USERAGENT": "Mozilla/5.0 (iPhone; CPU iPhone OS 18_7)",
        },
    ]
})
ir = ingest_r.json()
print(f"  status={ir['status']}  processed={ir['processed']}  failed={ir['failed']}")
# Inline recs are returned immediately in the response (core engine result)
for rec in ir.get("recommendations", []):
    print(f"  inline recs → user={rec['id']}  strategy={rec['strategy']}  count={rec['count']}  intent={rec['intent_score']}")
print()

# ── Step 2: Engine → Marketplace (push recommendations) ─────────────────────
print("Step 2: engine pushes recommendations to staging marketplace")
push_r = requests.post(f"{BASE}/api/v1/feed/push", headers=HEADERS, json={
    "account_id": ACCOUNT,
    "top_taxons": 3,
    "top_n_per_taxon": 10,
    "shop_feed_url": STAGING_PUSH,
    "push_timeout_seconds": 5.0,
})
pr = push_r.json()
print(f"  strategy={pr['strategy']}  total_products={pr['total_products']}")
print(f"  push_status={pr['push_status']}  push_url={pr.get('push_url','')[:60]}")
if pr.get("push_error"):
    print(f"  push_error={pr['push_error']}")
print()
for tf in pr.get("taxon_feeds", []):
    print(f"  [{tf['taxon_name'] or tf['taxon_id']}]  {tf['count']} products  score={tf['score']}")

# ── Step 3: Confirm logs ─────────────────────────────────────────────────────
print()
print("Step 3: audit logs")
push_log = requests.get(f"{BASE}/api/v1/logs/push?limit=5").json()
for e in push_log["entries"][:3]:
    sym = "✓" if e["push_status"] == "ok" else "✗"
    print(f"  {sym} {e['ts'][:19]}  account={e['account_id']}  products={e['products_count']}  status={e['push_status']}")


In [ ]:

# ── Ingestion diagnostics — run any time to check staging ingest health ───────
import requests, json, socket, time
from datetime import datetime, timezone

BASE = "http://localhost:8018"

print("=" * 60)
print(f"  TOKI Staging Ingestion Diagnostics  {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}")
print("=" * 60)

# 1. Server health
h = requests.get(f"{BASE}/api/v1/health", timeout=5).json()
print(f"\n[1] Server  : {h['status']}  env={h['environment']}  catalog={h['catalog_ready']}")

# 2. Oracle connectivity
for host, port, label in [("10.21.66.107", 1521, "Oracle DWH"), ("10.21.67.188", 5432, "PostgreSQL")]:
    try:
        s = socket.create_connection((host, port), timeout=3); s.close()
        print(f"[2] {label} {host}:{port}  REACHABLE")
    except Exception as e:
        print(f"[2] {label} {host}:{port}  UNREACHABLE — {e}")

# 3. Oracle poller checkpoint
try:
    with open("/workspaces/marketplace-stream-data-recommendation-engine/logs/.oracle_poll_checkpoint.json") as f:
        cp = json.load(f)
    ts = cp.get("consumer_events_ts", "never")
    pdate = cp.get("consumer_events_pdate", "?")
    print(f"[3] Oracle checkpoint : pdate={pdate}  last_ts={str(ts)[:25]}")
except FileNotFoundError:
    print("[3] Oracle checkpoint : NOT FOUND (poller never ran successfully)")

# 4. Live metrics
m = requests.get(f"{BASE}/api/v1/metrics", timeout=5).json()
ig = m.get("ingestion", {})
sys_m = m.get("system", {})
print(f"\n[4] Uptime            : {m.get('uptime_seconds', 0):.0f}s")
print(f"    events_processed  : {ig.get('events_processed', 0):,}")
print(f"    consumer_events   : {ig.get('consumer_events_processed', 0):,}")
print(f"    by_activity       : {ig.get('by_activity', {})}")
print(f"    tracked_users     : {sys_m.get('tracked_users', 0)}")
print(f"    active_sessions   : {sys_m.get('active_sessions', 0)}")

# 5. Recent ingest batches
log = requests.get(f"{BASE}/api/v1/logs/ingest?limit=8", timeout=5).json()
print(f"\n[5] Recent ingest batches (total_stored={log.get('total_stored',0):,}):")
for e in log["entries"][:6]:
    print(f"    {e['ts'][:19]}  [{e['source']:<16}]  processed={e['processed']:>4}  {e.get('event_types',{})}")

# 6. Oracle: rows available since checkpoint
try:
    import sys; sys.path.insert(0, "/workspaces/marketplace-stream-data-recommendation-engine")
    from src.module.database import oracle_import
    pdate_filter = cp.get("consumer_events_pdate", "20260819")
    ts_filter = str(cp.get("consumer_events_ts", ""))[:19].replace(" ", "T").rstrip("Z")
    df = oracle_import(
        f"SELECT COUNT(*) AS CNT FROM toki.marketplace_consumer_EVENTS "
        f"WHERE P_DATE >= '{pdate_filter}' AND SUBSTR(TIMESTAMP_,1,19) > '{ts_filter}'"
    )
    pending = int(df["CNT"].values[0])
    print(f"\n[6] Oracle rows pending ingest : {pending:,}  (since ts={ts_filter[:19]})")
    if pending > 0:
        print(f"    ✓ Oracle data available — next poll cycle will ingest these rows")
    else:
        print(f"    — No new rows yet (up to date)")
except Exception as e:
    print(f"\n[6] Oracle row check failed: {e}")

# 7. Recent push log
push = requests.get(f"{BASE}/api/v1/logs/push?limit=5", timeout=5).json()
print(f"\n[7] Recent marketplace pushes (total={push.get('total_stored',0):,}):")
for e in push["entries"][:4]:
    sym = "✓" if e["push_status"] == "ok" else ("↷" if e["push_status"] == "skipped" else "✗")
    print(f"    {e['ts'][:19]}  {sym} {e['push_status']:<8}  account={e['account_id'][:16]}  products={e['products_count']}")

print("\n" + "=" * 60)


In [ ]:

# ── Ingestion data quality: what the marketplace is actually sending ───────────
import json, ast, sys
from pathlib import Path
from collections import Counter

sys.path.insert(0, "/workspaces/marketplace-stream-data-recommendation-engine")

entries = []
for p in Path("/workspaces/marketplace-stream-data-recommendation-engine/logs").glob("toki_event_log_*.jsonl"):
    for line in p.read_text().splitlines():
        if line.strip():
            try: entries.append(json.loads(line))
            except: pass

entries.sort(key=lambda x: x.get("ts", ""), reverse=True)
print(f"Total ingested events in log: {len(entries):,}\n")

# ── Breakdown by source and event type ───────────────────────────────────────
sources   = Counter(e.get("source", "?") for e in entries)
evt_types = Counter(e.get("event_name") or e.get("activity_name", "?") for e in entries)

print("By source:")
for k, v in sources.most_common():
    bar = "█" * int(v / max(sources.values()) * 30)
    print(f"  {k:<22} {v:>6,}  {bar}")

print("\nBy event type:")
for k, v in evt_types.most_common():
    bar = "█" * int(v / max(evt_types.values()) * 30)
    print(f"  {k:<30} {v:>6,}  {bar}")

# ── event_value / activity_data format samples ──────────────────────────────
print("\nEvent payload format per type (raw vs decoded):")
shown = set()
for e in entries[:500]:
    etype = e.get("event_name") or e.get("activity_name", "?")
    if etype in shown:
        continue
    shown.add(etype)
    raw = e.get("event_value") or e.get("activity_data")
    fmt = "json" if isinstance(raw, (dict, list)) else ("py-literal" if isinstance(raw, str) and raw.startswith("{") and "'" in raw else "json-str")
    if isinstance(raw, str):
        try:    decoded = ast.literal_eval(raw)
        except:
            try:    decoded = json.loads(raw)
            except: decoded = raw
    else:
        decoded = raw
    print(f"\n  [{etype}]  format={fmt}  source={e.get('source')}")
    print(f"    raw    : {str(raw)[:110]}")
    print(f"    decoded: {str(decoded)[:110]}")

# ── Unique products and taxon signal coverage ─────────────────────────────────
product_ids = set()
taxon_labels = Counter()
for e in entries:
    raw = e.get("event_value") or {}
    if isinstance(raw, str):
        try:    raw = ast.literal_eval(raw)
        except:
            try: raw = json.loads(raw)
            except: raw = {}
    if isinstance(raw, dict):
        for pid in (raw.get("productIds") or []):
            product_ids.add(pid)
        t = (raw.get("taxon") or {}).get("label", "")
        if t:
            taxon_labels[t] += 1
    ad = e.get("activity_data") or {}
    if isinstance(ad, dict) and ad.get("productid"):
        product_ids.add(ad["productid"])

print(f"\nUnique product IDs seen     : {len(product_ids):,}")
print(f"Unique taxon labels seen    : {len(taxon_labels):,}")
print(f"Unique accounts (log sample): {len({e.get('account_id') for e in entries if e.get('account_id')}):,}")

print("\nTop 15 taxon labels (from oracle-poll + consumer-events):")
for label, cnt in taxon_labels.most_common(15):
    bar = "█" * int(cnt / max(taxon_labels.values()) * 25)
    print(f"  {label:<35} {cnt:>5,}  {bar}")

# ── Signal richness: what the engine can use from these events ─────────────────
print("\nSignal value by event type:")
WEIGHTS = {
    "order-events":   "★★★★★  5.0  (purchase completion)",
    "limit-events":   "★★★★   4.0  (credit check — strong intent)",
    "cart-events":    "★★★½   3.5  (add to cart)",
    "wishlist-events":"★★★    3.0  (wishlist add)",
    "product_click":  "★½      1.5  (explicit product click)",
    "view_product":   "★       1.0  (product page view)",
    "taxon_click":    "½       0.5  (category browse — weakest signal)",
}
for et, cnt in evt_types.most_common():
    weight = WEIGHTS.get(et, "—")
    print(f"  {et:<22} {cnt:>6,}  {weight}")
